# Notebook 01: Texas energy infrastructure, loading and cleaning

In [1]:
#Date : 10/15
# Data downloading
# Wells and pipeline data downloaded from: https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/
# Winds downloaded in GEOSON from:https://eerscmap.usgs.gov/uswtdb/data/
# Solar downlaoded in GEOSON from:
# battery storage:For the battery storage layer of our Texas energy footprint map, you should download the 2025 EIA-860 annual data from https://www.eia.gov/electricity/data/eia860/

My plan is to do the analysis this way:
Stage 1: Download and standardize
Download the RRC, USGS, EIA, and Census datasets. Standardize coordinate systems, identifiers, facility categories, and dates. Preserve the original data separately.
Stage 2: Spatial analysis
Use GeoPandas to assign facilities to counties, clip pipeline geometries at county boundaries, calculate facility density, and measure solar polygons.
Stage 3: Summary statistics
Create a county-by-county table containing facility counts, pipeline mileage, capacity, mapped acreage, and estimated land disturbance, with source and uncertainty information.
Stage 4: Maps and graphics
Generate an interactive map with Folium, statistical charts with Plotly, and static maps for with GeoPandas and Matplotlib.
The central output would be a county-level dataset that could be reused for the report, graphics
I'll see what can I do

In this notebook I'll Load the original USGS, EIA, RRC, and Census datasets. Inspect records, clean identifiers, standardize CRS and facility classifications, validate geometries, and save the cleaned datasets to 03_Cleaned_Data/.

## 1. Loading data

### 1.1. Defining the path and loadig data that I have so far

In [2]:
import pandas as pd
import geopandas as gpd

# main folder 

file_path = "G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint/"

# original datasets.

raw_data_path = file_path + "01_Raw_Data/"

#  the USGS wind turbine dataset.
# Note: I use the September 2026 version of the database.

wind_path = raw_data_path + "USGS_Wind/uswtdbGeoJSON/uswtdb_V9_1_20260928.geojson"

#  EIA plant dataset.
# note: I use the 2025 annual EIA-860 data.

battery_plant_path = raw_data_path + "EIA_Battery/eia8602025/2___Plant_Y2025.xlsx"

# EIA generator dataset.

battery_generator_path = raw_data_path + "EIA_Battery/eia8602025/3_1_Generator_Y2025.xlsx"

### 1.2. Laoding wind data

In [3]:
# I load the USGS wind turbine dataset using GeoPandas.
# GeoJSON contains geographic coordinates, so I can use
# it directly for spatial analysis and mapping.

wind_all = gpd.read_file(wind_path)

#the total number of wind turbines in the dataset.
print("Total U.S. wind turbines:", len(wind_all))

# check the number of columns.
print("Number of columns:", len(wind_all.columns))

# display the column names to understand the available data.
print("\nColumn names:")
print(wind_all.columns.tolist())

# I inspect the first five records.
display(wind_all.head())

Total U.S. wind turbines: 77839
Number of columns: 15

Column names:
['case_id', 't_state', 'p_name', 'p_year', 't_manu', 't_model', 't_cap', 't_hh', 't_rd', 't_rsa', 't_ttlh', 't_conf_atr', 't_conf_loc', 't_img_date', 'geometry']


,case_id,t_state,p_name,p_year,t_manu,t_model,t_cap,t_hh,t_rd,t_rsa,t_ttlh,t_conf_atr,t_conf_loc,t_img_date,geometry
0,3123493,OK,25 Mile Creek,2022.0,Vestas,V150-4.2,4200.0,105.0,150.0,17671.46,180.1,3,3,2023-01-04,POINT (-99.75696 36.49125)
1,3123580,OK,25 Mile Creek,2022.0,Vestas,V150-4.2,4200.0,105.0,150.0,17671.46,180.1,3,3,2023-01-04,POINT (-99.79066 36.44141)
2,3123616,OK,25 Mile Creek,2022.0,Vestas,V150-4.2,4200.0,105.0,150.0,17671.46,180.1,3,3,2023-01-04,POINT (-99.73301 36.48947)
3,3123278,OK,25 Mile Creek,2022.0,Vestas,V150-4.2,4200.0,105.0,150.0,17671.46,180.1,3,3,2023-01-04,POINT (-99.76085 36.42967)
4,3123340,OK,25 Mile Creek,2022.0,Vestas,V150-4.2,4200.0,105.0,150.0,17671.46,180.1,3,3,2023-01-04,POINT (-99.73309 36.45159)


In [4]:
# filtering the national wind turbine dataset to include
# only turbines located in Texas.

wind_tx = wind_all[
    wind_all["t_state"] == "TX"
].copy()

# I check how many wind turbines are recorded in Texas.
print("Total Texas wind turbines:", len(wind_tx))

# I check whether turbine identifiers are unique.
print(
    "Duplicate turbine IDs:",
    wind_tx["case_id"].duplicated().sum()
)

# I check how many turbines have missing geographic locations.
print(
    "Missing geometries:",
    wind_tx.geometry.isna().sum()
)

# I check the coordinate reference system.
print("Coordinate system:", wind_tx.crs)

# checking how many turbines are missing an installation year.
print(
    "Missing installation years:",
    wind_tx["p_year"].isna().sum()
)

# I check the number of unique wind project names.
print(
    "Unique project names:",
    wind_tx["p_name"].nunique()
)

#the distribution of installation years.
print("\nTurbines by installation year:")
print(
    wind_tx["p_year"]
    .value_counts(dropna=False)
    .sort_index()
)

# I inspect the first five Texas turbine records.
display(wind_tx.head())

Total Texas wind turbines: 19943
Duplicate turbine IDs: 0
Missing geometries: 0
Coordinate system: EPSG:4269
Missing installation years: 238
Unique project names: 294

Turbines by installation year:
p_year
1999.0      46
2001.0     845
2003.0      40
2005.0     109
2006.0      13
2007.0     717
2008.0    1413
2009.0    1326
2010.0     353
2011.0     136
2012.0     918
2013.0      84
2014.0     963
2015.0    1795
2016.0    1211
2017.0    1949
2018.0     915
2019.0    1496
2020.0    1394
2021.0    1065
2022.0    1337
2023.0     335
2024.0     404
2025.0     507
2026.0     334
NaN        238
Name: count, dtype: int64


,case_id,t_state,p_name,p_year,t_manu,t_model,t_cap,t_hh,t_rd,t_rsa,t_ttlh,t_conf_atr,t_conf_loc,t_img_date,geometry
901,3106719,TX,Amadeus,2020.0,GE Wind,GE2.82-127,2820.0,89.0,127.0,12667.69,152.4,3,3,2022-02-02,POINT (-100.51244 33.05536)
902,3107777,TX,Amadeus,2020.0,GE Wind,GE2.82-127,2820.0,89.0,127.0,12667.69,152.4,3,3,2021-01-28,POINT (-100.5514 33.02129)
903,3106560,TX,Amadeus,2020.0,GE Wind,GE1.79-100,1790.0,80.0,100.0,7853.98,130.1,3,3,2020-10-31,POINT (-100.58499 32.89265)
904,3107993,TX,Amadeus,2020.0,GE Wind,GE2.82-127,2820.0,89.0,127.0,12667.69,152.4,3,3,2021-01-28,POINT (-100.5294 32.96099)
905,3106744,TX,Amadeus,2020.0,GE Wind,GE1.79-100,1790.0,80.0,100.0,7853.98,130.1,3,3,2020-10-31,POINT (-100.57402 32.90321)


In [5]:
# defining the folder where I save my cleaned datasets.

wind_output_path = file_path + "03_Cleaned_Data/"

# I convert the coordinates from NAD83 (EPSG:4269)
# to WGS84 (EPSG:4326) for interactive web mapping.
# I keep all turbine records and their original attributes.

wind_tx_map = wind_tx.to_crs("EPSG:4326")

# saving the Texas wind turbine dataset as GeoJSON.

wind_tx_map.to_file(
    wind_output_path + "Texas_Wind_Turbines.geojson",
    driver="GeoJSON"
)

# Creating a separate CSV version for statistical analysis.
# I extract longitude and latitude from the geometry.

wind_tx_csv = wind_tx_map.copy()

wind_tx_csv["longitude"] = wind_tx_csv.geometry.x
wind_tx_csv["latitude"] = wind_tx_csv.geometry.y

# I remove the geometry column because I already have
# longitude and latitude in the CSV file.

wind_tx_csv = wind_tx_csv.drop(columns="geometry")

#saving

wind_tx_csv.to_csv(
    wind_output_path + "Texas_Wind_Turbines.csv",
    index=False
)


print("GeoJSON saved successfully.")
print("CSV saved successfully.")
print("Total Texas wind turbines:", len(wind_tx_map))

GeoJSON saved successfully.
CSV saved successfully.
Total Texas wind turbines: 19943


### 1.3.Loading Solar data

In [6]:
# I define the path to the USGS solar dataset.
# I use version 4.0, released on April 14, 2026.

solar_path = raw_data_path + "USGS_Solar/uspvdbGeoJSON/uspvdb_v4_0_20260414.geojson"

# load the solar dataset using GeoPandas.
solar_all = gpd.read_file(solar_path)

# check the total number of solar records.
print("Total U.S. solar records:", len(solar_all))

# check the number of columns.
print("Number of columns:", len(solar_all.columns))

# display the column names.
print("\nColumn names:")
print(solar_all.columns.tolist())

# I inspect the first five records.
display(solar_all.head())

Total U.S. solar records: 6611
Number of columns: 29

Column names:
['FID', 'Id', 'case_id', 'multi_poly', 'eia_id', 'p_state', 'p_county', 'ylat', 'xlong', 'p_area', 'p_img_date', 'p_dig_conf', 'p_name', 'p_year', 'p_pwr_reg', 'p_tech_pri', 'p_tech_sec', 'p_sys_type', 'p_axis', 'p_azimuth', 'p_tilt', 'p_battery', 'p_cap_ac', 'p_cap_dc', 'p_type', 'p_agrivolt', 'p_comm', 'p_zscore', 'geometry']


,FID,Id,case_id,multi_poly,eia_id,p_state,p_county,ylat,xlong,p_area,...,p_azimuth,p_tilt,p_battery,p_cap_ac,p_cap_dc,p_type,p_agrivolt,p_comm,p_zscore,geometry
0,0,0,400011,single,2953,OK,Canadian,35.4774,-97.6764,50723,...,180.0,25.0,None,2.5,3.0,greenfield,non-agrivoltaic,community solar,-0.219022,"POLYGON ((-97.6768 35.47837, -97.67701 35.4783..."
1,1,0,407013,single,67406,NY,Erie,43.0168,-78.9779,77051,...,180.0,52.0,None,4.0,5.5,greenfield,non-agrivoltaic,community solar,-0.048823,"POLYGON ((-78.97948 43.01831, -78.97944 43.015..."
2,2,0,404351,single,64787,IL,Macoupin,39.4045,-90.0025,56943,...,25.0,NaN,None,2.0,3.0,greenfield,non-agrivoltaic,community solar,-0.308881,"POLYGON ((-90.00022 39.40521, -90.00142 39.405..."
3,3,0,404352,single,64788,NY,Greene,42.3708,-74.0406,40085,...,25.0,NaN,None,2.0,3.0,greenfield,non-agrivoltaic,community solar,-0.000703,"POLYGON ((-74.04066 42.3727, -74.04092 42.3727..."
4,4,0,404336,single,64734,IL,Stephenson,42.3804,-89.8412,55382,...,25.0,NaN,None,2.0,3.0,greenfield,non-agrivoltaic,community solar,-0.288227,"POLYGON ((-89.84004 42.38148, -89.84265 42.381..."


In [7]:
#checking solar infos:
# I filter the national solar dataset to include
# only solar facilities located in Texas.

solar_tx = solar_all[
    solar_all["p_state"] == "TX"
].copy()

# I check how many solar records are located in Texas.
print("Total Texas solar records:", len(solar_tx))

# I check how many unique solar identifiers are present.
print(
    "Unique solar case IDs:",
    solar_tx["case_id"].nunique()
)

# I check whether any solar case IDs appear more than once.
print(
    "Duplicate case IDs:",
    solar_tx["case_id"].duplicated().sum()
)

# I check the number of unique project names.
print(
    "Unique project names:",
    solar_tx["p_name"].nunique()
)

# I check the geographic coordinate system.
print("Coordinate system:", solar_tx.crs)

# I check the geometry types.
print("\nGeometry types:")
print(solar_tx.geometry.geom_type.value_counts())

# I check for missing and empty geometries.
print(
    "\nMissing geometries:",
    solar_tx.geometry.isna().sum()
)

print(
    "Empty geometries:",
    solar_tx.geometry.is_empty.sum()
)

# I check how many records are missing installation years.
print(
    "Missing installation years:",
    solar_tx["p_year"].isna().sum()
)

# I inspect the first five Texas solar records.
display(solar_tx.head())

Total Texas solar records: 187
Unique solar case IDs: 187
Duplicate case IDs: 0
Unique project names: 187
Coordinate system: EPSG:4326

Geometry types:
MultiPolygon    152
Polygon          35
Name: count, dtype: int64

Missing geometries: 0
Empty geometries: 0
Missing installation years: 0


,FID,Id,case_id,multi_poly,eia_id,p_state,p_county,ylat,xlong,p_area,...,p_azimuth,p_tilt,p_battery,p_cap_ac,p_cap_dc,p_type,p_agrivolt,p_comm,p_zscore,geometry
189,189,0,403279,multi,62932,TX,Ector,31.6943,-102.6884,708041,...,165.0,52.0,None,30.0,37.3,greenfield,non-agrivoltaic,community solar,-0.308933,"MULTIPOLYGON (((-102.69542 31.69015, -102.6957..."
239,239,0,406516,multi,67284,TX,Jackson,29.1121,-96.5242,3096642,...,181.0,NaN,batteries,150.5,208.5,greenfield,non-agrivoltaic,None,-0.105159,"MULTIPOLYGON (((-96.53071 29.11304, -96.53083 ..."
312,312,0,406915,multi,67006,TX,Pecos,31.0289,-102.4841,5576807,...,180.0,0.0,None,306.0,398.0,greenfield,non-agrivoltaic,None,-0.049022,"MULTIPOLYGON (((-102.49295 31.03596, -102.4929..."
427,427,0,407272,multi,68246,TX,Milam,30.9038,-97.0566,596296,...,NaN,NaN,None,40.0,40.0,greenfield,non-agrivoltaic,None,-0.108639,"MULTIPOLYGON (((-97.05905 30.90067, -97.05914 ..."
428,428,0,406937,multi,67058,TX,Tarrant,32.8314,-97.3494,39211,...,NaN,NaN,None,4.0,4.0,greenfield,non-agrivoltaic,None,0.377223,"MULTIPOLYGON (((-97.34724 32.83105, -97.34754 ..."


In [8]:
# I check whether all solar polygons have valid geometries.
# Invalid geometries could affect land-area calculations.

print(
    "Valid geometries:",
    solar_tx.geometry.is_valid.sum()
)

print(
    "Invalid geometries:",
    (~solar_tx.geometry.is_valid).sum()
)

# I check whether any solar polygons have zero area.
# I project the data into a coordinate system measured
# in meters before calculating area.

solar_area_check = solar_tx.to_crs("EPSG:3083").copy()

solar_area_check["calculated_area_m2"] = (
    solar_area_check.geometry.area
)

print(
    "Zero-area geometries:",
    (solar_area_check["calculated_area_m2"] == 0).sum()
)

Valid geometries: 186
Invalid geometries: 1
Zero-area geometries: 0


In [9]:
# I need a library that checks geometry against geometric validity rules and explains what is wrong if the geometry is invalid.
#For example,a solar farm represented by a polygon.
# I want to know if that solar is valid or invalid polygon for example does the boundry cross itself?Shapely can identify this problem
# explain_validity explains specifically why a geometry is invalid
from shapely.validation import explain_validity

# I identify solar records with invalid geometries.
# I want to understand the problem before making any changes.

invalid_solar = solar_tx[
    ~solar_tx.geometry.is_valid
].copy()

# I identify why each geometry is invalid.
invalid_solar["geometry_issue"] = (
    invalid_solar.geometry.apply(explain_validity)
)

# I display the affected project and the geometry problem.
display(
    invalid_solar[
        [
            "case_id",
            "p_name",
            "p_county",
            "p_area",
            "geometry_issue"
        ]
    ]
)

,case_id,p_name,p_county,p_area,geometry_issue
4607,403557,Aragorn Solar Project,Culberson,4934850,Ring Self-intersection[-104.463721844866 31.73...


In [10]:
# Now I need to fix that
# I import make_valid from the Shapely library.
# Shapely is a Python library for working with geographic shapes.
# The make_valid function attempts to repair invalid polygons,
# such as polygons whose boundaries intersect themselves.

from shapely.validation import make_valid

# I create a copy of my Texas solar dataset because I want
# to preserve the original geometries for comparison.
# I will make any corrections only in this new dataset.

solar_tx_clean = solar_tx.copy()

# I identify the geometries that do not follow valid
# polygon rules. In my dataset, I found one solar project
# with a ring self-intersection.

invalid_mask = ~solar_tx_clean.geometry.is_valid

# I apply make_valid only to the invalid geometry.
# This function attempts to reconstruct the polygon into
# a valid geographic shape without unnecessarily changing
# the rest of the dataset.
# I do not modify geometries that are already valid.

solar_tx_clean.loc[invalid_mask, "geometry"] = (
    solar_tx_clean.loc[invalid_mask, "geometry"]
    .apply(make_valid)
)

# check whether the repair was successful.
# I expect the number of invalid geometries to be zero.

print(
    "Invalid geometries after repair:",
    (~solar_tx_clean.geometry.is_valid).sum()
)

# I calculate the original and repaired polygon areas.
# I first convert the geometries to EPSG:3083,
# a projected coordinate system suitable for Texas.
# This allows me to calculate area in square meters
# rather than using geographic coordinates in degrees.

original_area = (
    solar_tx.loc[invalid_mask]
    .to_crs("EPSG:3083")
    .geometry.area
)

repaired_area = (
    solar_tx_clean.loc[invalid_mask]
    .to_crs("EPSG:3083")
    .geometry.area
)

# I compare the original and repaired areas to see
# whether fixing the geometry changed the mapped footprint.

print("\nOriginal calculated areas (m²):", original_area.to_list())
print("Repaired calculated areas (m²):", repaired_area.to_list())

# I calculate the percentage change in area.
# A small difference would suggest that the repair
# preserved the original mapped footprint.

print(
    "Percentage area change:",
    (
        (repaired_area.sum() - original_area.sum())
        / original_area.sum() if original_area.sum() else float("nan")
    ) * 100
)

# I check the geometry type after the repair.
# Some repairs can change a Polygon or MultiPolygon
# into a GeometryCollection, which may require
# additional processing before mapping.

print(
    "\nRepaired geometry type:",
    solar_tx_clean.loc[invalid_mask].geometry.geom_type.value_counts().to_dict()
)

Invalid geometries after repair: 0

Original calculated areas (m²): [4934852.262097784]
Repaired calculated areas (m²): [4934852.262097785]
Percentage area change: 1.887234967029342e-14

Repaired geometry type: {'MultiPolygon': 1}


In [11]:
# I compare the area reported in the USGS dataset
# with the area calculated from the repaired polygons.
# I use EPSG:3083 so the calculated areas are in square meters.

solar_area_check = solar_tx_clean.to_crs("EPSG:3083").copy()

# I calculate the area of each solar polygon in square meters.

solar_area_check["calculated_area_m2"] = (
    solar_area_check.geometry.area
)

# I convert the reported area column to numeric values
# so I can compare it with my calculated areas.

solar_area_check["reported_area"] = pd.to_numeric(
    solar_area_check["p_area"],
    errors="coerce"
)

# I compare reported and calculated area only after confirming that
# p_area uses square meters in the source documentation.
# I do not interpret a percentage difference until the units are verified.
# I calculate the percentage difference between the
# calculated and reported areas.
# I exclude zero or missing reported areas to avoid
# dividing by zero or producing misleading results.

solar_area_check["area_difference_pct"] = pd.NA

# I set this to True only after verifying the USGS p_area units.
SOLAR_REPORTED_AREA_IS_M2_VERIFIED = False
valid_area_mask = (solar_area_check["reported_area"] > 0) & SOLAR_REPORTED_AREA_IS_M2_VERIFIED

solar_area_check.loc[valid_area_mask, "area_difference_pct"] = (
    (
        solar_area_check.loc[valid_area_mask, "calculated_area_m2"]
        - solar_area_check.loc[valid_area_mask, "reported_area"]
    )
    / solar_area_check.loc[valid_area_mask, "reported_area"]
) * 100

# I summarize the percentage differences.

print("Area difference summary:")
print(solar_area_check["area_difference_pct"].describe())

# I identify projects with area differences greater than 5%.

large_differences = solar_area_check[
    solar_area_check["area_difference_pct"].abs() > 5
]

print(
    "\nProjects with area differences greater than 5%:",
    len(large_differences)
)

# I display the projects with the largest differences
# so I can investigate any unexpected results.

display(
    large_differences[
        [
            "case_id",
            "p_name",
            "p_county",
            "p_area",
            "calculated_area_m2",
            "area_difference_pct"
        ]
    ].head(10)
)

Area difference summary:
count       0
unique      0
top       NaN
freq      NaN
Name: area_difference_pct, dtype: object

Projects with area differences greater than 5%: 0


,case_id,p_name,p_county,p_area,calculated_area_m2,area_difference_pct


In [12]:
# I convert the percentage difference column to numeric values
# so I can calculate meaningful summary statistics.

solar_area_check["area_difference_pct"] = pd.to_numeric(
    solar_area_check["area_difference_pct"],
    errors="coerce"
)

# I check the distribution of area differences.
print("Area difference summary:")
print(solar_area_check["area_difference_pct"].describe())

# I identify the largest absolute percentage difference
# to confirm how closely the calculated areas match
# the reported areas.

print(
    "\nLargest absolute percentage difference:",
    solar_area_check["area_difference_pct"].abs().max()
)

# I check for missing or nonpositive reported areas.
print(
    "Missing reported areas:",
    solar_area_check["reported_area"].isna().sum()
)

print(
    "Nonpositive reported areas:",
    (solar_area_check["reported_area"] <= 0).sum()
)

Area difference summary:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: area_difference_pct, dtype: float64

Largest absolute percentage difference: nan
Missing reported areas: 0
Nonpositive reported areas: 0


In [13]:
# Beautiful
# I define the folder where I save my cleaned datasets.

solar_output_path = file_path + "03_Cleaned_Data/"

# I save the Texas solar dataset with the repaired geometry.
# I keep the original USGS attributes, including the reported area.
# The data is already in EPSG:4326, which works for interactive maps.

solar_tx_clean.to_file(
    solar_output_path + "Texas_Solar_Projects.geojson",
    driver="GeoJSON"
)

# I create a separate CSV version for statistical analysis.
# I keep the original latitude and longitude columns
# provided by USGS and remove the geometry column.

solar_tx_csv = solar_tx_clean.drop(columns="geometry")

# I save the CSV file.

solar_tx_csv.to_csv(
    solar_output_path + "Texas_Solar_Projects.csv",
    index=False
)

# I confirm that the files have been saved.

print("Solar GeoJSON saved successfully.")
print("Solar CSV saved successfully.")
print("Total Texas solar records:", len(solar_tx_clean))

Solar GeoJSON saved successfully.
Solar CSV saved successfully.
Total Texas solar records: 187


In [14]:
# I also save it in csv
# I keep all original attributes and remove the geometry column.

solar_tx_csv = solar_tx_clean.drop(columns="geometry")

# I save the complete solar dataset as a CSV.

solar_tx_csv.to_csv(
    solar_output_path + "Texas_Solar_Projects.csv",
    index=False
)

# I confirm that the CSV has been saved.

print("Texas solar CSV saved successfully.")
print("Total solar records:", len(solar_tx_csv))

Texas solar CSV saved successfully.
Total solar records: 187


### 1.4. Loading battery data

In [15]:
# I define the paths to the EIA-860 plant and generator files.
# I use the 2025 annual dataset to identify battery
# storage facilities in Texas.

battery_plant_path = raw_data_path + "EIA_Battery/eia8602025/2___Plant_Y2025.xlsx"

battery_generator_path = raw_data_path + "EIA_Battery/eia8602025/3_1_Generator_Y2025.xlsx"

# I check the sheet names before loading the data.
# EIA Excel files sometimes contain multiple sheets,
# including documentation and data tables.

plant_excel = pd.ExcelFile(battery_plant_path)
generator_excel = pd.ExcelFile(battery_generator_path)

print("Plant file sheets:")
print(plant_excel.sheet_names)

print("\nGenerator file sheets:")
print(generator_excel.sheet_names)

Plant file sheets:
['Plant']

Generator file sheets:
['Operable', 'Proposed', 'Retired and Canceled']


In [16]:
#EIA files often include a descriptive row above the actual column headers. 
# I inspect the first five rows of the EIA plant file.
# I load the sheet without assigning column headers
# so I can identify where the actual table begins.

plant_preview = pd.read_excel(
    battery_plant_path,
    sheet_name="Plant",
    header=None,
    nrows=5
)

# I inspect the first five rows of the operable generator file.
# I focus on operable generators because I want to identify
# battery storage facilities that are reported as operating.

generator_preview = pd.read_excel(
    battery_generator_path,
    sheet_name="Operable",
    header=None,
    nrows=5
)

print("Plant file preview:")
display(plant_preview)

print("\nOperable generator file preview:")
display(generator_preview)

Plant file preview:


,0,1,2,3,4,5,6,7,8,9,...,32,33,34,35,36,37,38,39,40,41
0,"2025 Form EIA-860 Data - Schedule 2, 'Plant Data'",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Utility ID,Utility Name,Plant Code,Plant Name,Street Address,City,State,Zip,County,Latitude,...,Grid Voltage 2 (kV),Grid Voltage 3 (kV),Energy Storage,Natural Gas LDC Name,Natural Gas Pipeline Name 1,Natural Gas Pipeline Name 2,Natural Gas Pipeline Name 3,Pipeline Notes,Natural Gas Storage,Liquefied Natural Gas Storage
2,63560,"Sand Point Generating, LLC",1,Sand Point,100 Power Plant Way,Sand Point,AK,99661,Aleutians East,55.339722,...,,,N,NaN,NaN,NaN,NaN,NaN,N,X
3,195,Alabama Power Co,2,Bankhead Dam,19001 Lock 17 Road,Northport,AL,35476,Tuscaloosa,33.458665,...,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,195,Alabama Power Co,3,Barry,North Highway 43,Bucks,AL,36512,Mobile,31.0069,...,,,N,NaN,BAY GAS STORAGE,NaN,NaN,NaN,N,X



Operable generator file preview:


,0,1,2,3,4,5,6,7,8,9,...,63,64,65,66,67,68,69,70,71,72
0,"2025 Form EIA-860 Data - Schedule 3, 'Generato...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Technology,Prime Mover,Unit Code,...,Planned Energy Source 1,Planned New Nameplate Capacity (MW),Planned Repower Month,Planned Repower Year,Other Planned Modifications?,Other Modifications Month,Other Modifications Year,Multiple Fuels?,Cofire Fuels?,Switch Between Oil and Natural Gas?
2,63560,"Sand Point Generating, LLC",1,Sand Point,AK,Aleutians East,1,Petroleum Liquids,IC,NaN,...,NaN,,,,NaN,,,N,NaN,NaN
3,63560,"Sand Point Generating, LLC",1,Sand Point,AK,Aleutians East,2,Petroleum Liquids,IC,NaN,...,NaN,,,,NaN,,,N,NaN,NaN
4,63560,"Sand Point Generating, LLC",1,Sand Point,AK,Aleutians East,3,Petroleum Liquids,IC,NaN,...,NaN,,,,NaN,,,N,NaN,NaN


In [17]:
# I load the EIA-860 plant dataset.
# I skip the first row because it contains the dataset title,
# not the column names.

battery_plants = pd.read_excel(
    battery_plant_path,
    sheet_name="Plant",
    skiprows=1
)

# I load the operable generator dataset.
# I use the Operable sheet because I want to focus
# on generators reported as operating in 2025.

battery_generators = pd.read_excel(
    battery_generator_path,
    sheet_name="Operable",
    skiprows=1
)

# I check the number of records and columns in each dataset.

print("Plant records:", len(battery_plants))
print("Plant columns:", len(battery_plants.columns))

print("\nOperable generator records:", len(battery_generators))
print("Generator columns:", len(battery_generators.columns))

# I check the available generator technologies.
# I want to identify how EIA classifies battery storage
# before filtering the data.

print("\nGenerator technologies:")
print(
    battery_generators["Technology"]
    .value_counts(dropna=False)
)

Plant records: 17349
Plant columns: 42

Operable generator records: 27919
Generator columns: 73

Generator technologies:
Technology
Solar Photovoltaic                             7999
Conventional Hydroelectric                     3981
Petroleum Liquids                              3807
Natural Gas Fired Combustion Turbine           2240
Natural Gas Fired Combined Cycle               1941
Natural Gas Internal Combustion Engine         1849
Onshore Wind Turbine                           1570
Landfill Gas                                   1211
Batteries                                      1003
Natural Gas Steam Turbine                       550
Conventional Steam Coal                         447
Wood/Wood Waste Biomass                         271
Other Waste Biomass                             199
Other Natural Gas                               177
Geothermal                                      163
Hydroelectric Pumped Storage                    150
Nuclear                             

In [18]:
# I filter the EIA operable generator dataset to include
# only battery generators located in Texas.
# I use the Technology column to identify battery storage.

battery_tx = battery_generators[
    (battery_generators["State"] == "TX")
    & (battery_generators["Technology"] == "Batteries")
].copy()

# I check the number of battery generator records in Texas.
print("Texas battery generator records:", len(battery_tx))

# I count unique plants using their EIA Plant Code.
# A single plant may have multiple battery generators,
# so the number of generators may be higher than
# the number of battery facilities.

print(
    "Unique Texas battery plants:",
    battery_tx["Plant Code"].nunique()
)

# I check whether generator identifiers are unique
# within each plant.

print(
    "Duplicate plant-generator combinations:",
    battery_tx.duplicated(
        subset=["Plant Code", "Generator ID"]
    ).sum()
)

# I check how many battery generators are reported
# at each plant.

print("\nBattery generators per plant:")
print(
    battery_tx.groupby("Plant Code")
    .size()
    .describe()
)

# I inspect the first five Texas battery records.
display(battery_tx.head())

Texas battery generator records: 205
Unique Texas battery plants: 202
Duplicate plant-generator combinations: 0

Battery generators per plant:
count    202.000000
mean       1.014851
std        0.121259
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        2.000000
dtype: float64


,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Technology,Prime Mover,Unit Code,...,Planned Energy Source 1,Planned New Nameplate Capacity (MW),Planned Repower Month,Planned Repower Year,Other Planned Modifications?,Other Modifications Month,Other Modifications Year,Multiple Fuels?,Cofire Fuels?,Switch Between Oil and Natural Gas?
8110,55983,Luminant Generation Company LLC,8063.0,DeCordova Steam Electric Station,TX,Hood,BESS,Batteries,BA,NaN,...,NaN,,,,NaN,,,NaN,NaN,NaN
14202,65411,Deriva Energy,56961.0,Notrees Windpower Hybrid,TX,Ector,BATT,Batteries,BA,NaN,...,NaN,,,,NaN,,,NaN,NaN,NaN
14234,67192,RWE Clean Energy,56981.0,Pyron Wind Farm LLC Hybrid,TX,Fisher,PYRBT,Batteries,BA,NaN,...,NaN,,,,NaN,,,N,NaN,NaN
14240,67192,RWE Clean Energy,56984.0,Inadale Wind Farm LLC Hybrid,TX,Nolan,INABT,Batteries,BA,NaN,...,NaN,,,,NaN,,,N,N,NaN
17066,58939,Cameron Wind 1 LLC,59118.0,Cameron Wind 1 LLC,TX,Cameron,SABAL,Batteries,BA,NaN,...,NaN,,,,NaN,,,NaN,NaN,NaN


Good. We have identified 205 battery generator records across 202 unique plants in Texas in the EIA-860 2025 Operable dataset.
The difference is small: most plants have one battery generator, while three plants have two. For our interactive map, we will eventually create one point per plant, but we will retain all 205 generator records when calculating capacity.
Before joining the datasets, I'll check the capacity fields and plant coordinates.

#### 1.4.1 Identify the available battery capacity columns

In [19]:
# I inspect the columns related to battery capacity and
# operating dates before choosing which values to analyze.
# I want to distinguish power capacity in MW from
# energy storage capacity in MWh.

battery_columns = [
    col for col in battery_tx.columns
    if any(
        term in col.lower()
        for term in ["capacity", "storage", "operating", "status"]
    )
]

print("Relevant battery columns:")
for col in battery_columns:
    print(col)

Relevant battery columns:
Nameplate Capacity (MW)
Summer Capacity (MW)
Winter Capacity (MW)
Status
Operating Month
Operating Year
Planned Net Summer Capacity Uprate (MW)
Planned Net Winter Capacity Uprate (MW)
Planned Net Summer Capacity Derate (MW)
Planned Net Winter Capacity Derate (MW)
Planned New Nameplate Capacity (MW)


Great. The EIA-860 generator dataset includes Nameplate Capacity (MW), which I can use to calculate the reported power capacity of Texas battery generators.

#### 1.4.2 Check battery capacity and operating status

In [20]:
# I convert battery nameplate capacity to numeric values
# so I can calculate totals and identify missing data.

battery_tx["Nameplate Capacity (MW)"] = pd.to_numeric(
    battery_tx["Nameplate Capacity (MW)"],
    errors="coerce"
)

# I check the distribution of battery power capacity.
print("Battery nameplate capacity (MW):")
print(battery_tx["Nameplate Capacity (MW)"].describe())

# I check whether any battery generators have missing,
# zero, or negative capacity values.

print(
    "\nMissing capacity values:",
    battery_tx["Nameplate Capacity (MW)"].isna().sum()
)

print(
    "Zero or negative capacity values:",
    (battery_tx["Nameplate Capacity (MW)"] <= 0).sum()
)

# I check generator status codes.
# Although I loaded the Operable sheet, I want to
# verify which operating statuses are represented.

print("\nGenerator status codes:")
print(battery_tx["Status"].value_counts(dropna=False))

# I examine the reported operating years.
print("\nBattery generators by operating year:")
print(
    battery_tx["Operating Year"]
    .value_counts(dropna=False)
    .sort_index()
)

Battery nameplate capacity (MW):
count    205.000000
mean      68.825854
std       77.630144
min        1.500000
25%        9.900000
50%       12.000000
75%      101.000000
max      304.000000
Name: Nameplate Capacity (MW), dtype: float64

Missing capacity values: 0
Zero or negative capacity values: 0

Generator status codes:
Status
OP    203
OS      1
OA      1
Name: count, dtype: int64

Battery generators by operating year:
Operating Year
2012.0     1
2017.0     1
2018.0     3
2019.0     2
2020.0    12
2021.0    17
2022.0    23
2023.0    34
2024.0    56
2025.0    56
Name: count, dtype: int64


I checked here: https://www.epa.gov/egrid/code-lookup

These results look good. We have 205 Texas battery generator records, and all have positive, nonmissing nameplate capacities.
I should investigate before calculating the final statewide total: the operating status codes.
OP generally means operating, OS means out of service, and OA means out of service but available for use. Since our map is intended to show operating battery facilities, we should distinguish these statuses rather than automatically treating all 205 generators as actively operating.
The operating-year data also show substantial recent growth: 112 of the 205 generator records have operating years of 2024 or 2025. That could be an interesting finding for our report, although it represents generator records, not necessarily distinct new facilities.

In [21]:
# I identify battery generators that are not classified
# as OP (operating).
# Although these records appear in EIA's Operable sheet,
# I want to distinguish operating generators from
# those reported as out of service.

battery_non_op = battery_tx[
    battery_tx["Status"] != "OP"
].copy()

# I inspect the affected generators and their capacities.
# I will use this information to decide which records
# should be included in my operating battery totals.

display(
    battery_non_op[
        [
            "Plant Code",
            "Plant Name",
            "County",
            "Generator ID",
            "Status",
            "Nameplate Capacity (MW)",
            "Operating Year"
        ]
    ]
)

# I compare the capacity of all battery generators
# with the capacity of generators classified as OP.

print(
    "Total capacity, all statuses (MW):",
    battery_tx["Nameplate Capacity (MW)"].sum()
)

print(
    "Total capacity, OP only (MW):",
    battery_tx.loc[
        battery_tx["Status"] == "OP",
        "Nameplate Capacity (MW)"
    ].sum()
)

,Plant Code,Plant Name,County,Generator ID,Status,Nameplate Capacity (MW),Operating Year
22146,63737.0,Chisholm Grid Energy Storage System,Tarrant,CHS01,OS,100.0,2021.0
27450,69352.0,"Citrus Flatts Energy Center, LLC",Cameron,CFBA1,OA,100.0,2023.0


Total capacity, all statuses (MW): 14109.3
Total capacity, OP only (MW): 13909.3


In [23]:
import os

# I preserve the complete Texas battery generator inventory for later scenarios.
# I use OP only for the operating-facility map and capacity totals below.
battery_inventory_folder = os.path.join(file_path, "03_Cleaned_Data", "Battery")
os.makedirs(battery_inventory_folder, exist_ok=True)
battery_tx.to_csv(os.path.join(battery_inventory_folder, "TX_Battery_All_Statuses.csv"), index=False)

# I keep only battery generators with OP status.
# I use this subset for my operating battery analysis,
# while preserving battery_tx with all original statuses.

battery_tx_op = battery_tx[
    battery_tx["Status"] == "OP"
].copy()

# I check how many unique plants have operating batteries.
# I count plants separately from generators because
# one plant can have more than one battery generator.

print("Operating battery generators:", len(battery_tx_op))
print(
    "Unique plants with operating batteries:",
    battery_tx_op["Plant Code"].nunique()
)

# I select the plant information needed for mapping.
# I use Plant Code to match generators to their
# corresponding EIA plant locations.

plant_locations = battery_plants[
    ["Plant Code", "Plant Name", "State", "County", "Latitude", "Longitude"]
].copy()

# I check whether plant identifiers are unique before merging.
# Duplicate plant codes could accidentally multiply
# the number of generator records.

print(
    "\nDuplicate plant codes in plant dataset:",
    plant_locations["Plant Code"].duplicated().sum()
)

# I merge operating battery generators with plant locations.
# I keep every battery generator even if its plant
# information is missing, so I can investigate unmatched records.

battery_tx_merged = battery_tx_op.merge(
    plant_locations,
    on="Plant Code",
    how="left",
    suffixes=("_generator", "_plant"),
    validate="many_to_one",
    indicator=True
)

# I check whether all operating battery generators
# matched a plant record.

print("\nMerge results:")
print(battery_tx_merged["_merge"].value_counts())

# I check whether any operating battery generators
# are missing geographic coordinates.

print(
    "\nMissing latitude:",
    battery_tx_merged["Latitude"].isna().sum()
)

print(
    "Missing longitude:",
    battery_tx_merged["Longitude"].isna().sum()
)

# I inspect the first five matched records.

display(
    battery_tx_merged[
        [
            "Plant Code",
            "Plant Name_plant",
            "County_plant",
            "Latitude",
            "Longitude",
            "Nameplate Capacity (MW)"
        ]
    ].head()
)

Operating battery generators: 203
Unique plants with operating batteries: 200

Duplicate plant codes in plant dataset: 0

Merge results:
_merge
both          203
left_only       0
right_only      0
Name: count, dtype: int64

Missing latitude: 0
Missing longitude: 0


,Plant Code,Plant Name_plant,County_plant,Latitude,Longitude,Nameplate Capacity (MW)
0,8063.0,DeCordova Steam Electric Station,Hood,32.403056,-97.700556,269.1
1,56961.0,Notrees Windpower Hybrid,Ector,31.995,-102.828333,36.0
2,56981.0,Pyron Wind Farm LLC Hybrid,Fisher,32.5886,-100.6728,9.9
3,56984.0,Inadale Wind Farm LLC Hybrid,Nolan,32.4969,-100.5797,9.9
4,59118.0,Cameron Wind 1 LLC,Cameron,26.195278,-97.466111,16.4


In [24]:
# I group operating battery generators by their EIA Plant Code.
# Some plants have more than one battery generator, so I add
# their nameplate capacities together to calculate the
# total operating battery capacity at each plant.

battery_tx_plants = (
    battery_tx_merged
    .groupby("Plant Code", as_index=False)
    .agg(
        plant_name=("Plant Name_plant", "first"),
        county=("County_plant", "first"),
        latitude=("Latitude", "first"),
        longitude=("Longitude", "first"),
        battery_generators=("Generator ID", "count"),
        capacity_mw=("Nameplate Capacity (MW)", "sum")
    )
)

# I check that each battery plant appears only once.
print("Total battery plants:", len(battery_tx_plants))

print(
    "Duplicate plant codes:",
    battery_tx_plants["Plant Code"].duplicated().sum()
)

# I compare the total capacity before and after aggregation.
# The totals should match because I am combining generators,
# not removing their capacity.

print(
    "\nGenerator-level capacity (MW):",
    battery_tx_op["Nameplate Capacity (MW)"].sum()
)

print(
    "Plant-level capacity (MW):",
    battery_tx_plants["capacity_mw"].sum()
)

# I check how many plants contain multiple battery generators.
print(
    "\nPlants with multiple battery generators:",
    (battery_tx_plants["battery_generators"] > 1).sum()
)

# I inspect the first five plant records.
display(battery_tx_plants.head())

Total battery plants: 200
Duplicate plant codes: 0

Generator-level capacity (MW): 13909.3
Plant-level capacity (MW): 13909.3

Plants with multiple battery generators: 3


,Plant Code,plant_name,county,latitude,longitude,battery_generators,capacity_mw
0,8063.0,DeCordova Steam Electric Station,Hood,32.403056,-97.700556,1,269.1
1,56961.0,Notrees Windpower Hybrid,Ector,31.995,-102.828333,1,36.0
2,56981.0,Pyron Wind Farm LLC Hybrid,Fisher,32.5886,-100.6728,1,9.9
3,56984.0,Inadale Wind Farm LLC Hybrid,Nolan,32.4969,-100.5797,1,9.9
4,59118.0,Cameron Wind 1 LLC,Cameron,26.195278,-97.466111,1,16.4


In [25]:
# I convert the battery plant table into a GeoDataFrame.
# I use each plant's longitude and latitude to create
# a geographic point for the interactive map.

battery_tx_map = gpd.GeoDataFrame(
    battery_tx_plants.copy(),
    geometry=gpd.points_from_xy(
        battery_tx_plants["longitude"],
        battery_tx_plants["latitude"]
    ),
    crs="EPSG:4326"
)

# I define the folder where I save my cleaned datasets.

battery_output_path = file_path + "03_Cleaned_Data/"

# I save the geographic dataset as GeoJSON.
# Each point represents one EIA plant with operating batteries.

battery_tx_map.to_file(
    battery_output_path + "Texas_Battery_Plants.geojson",
    driver="GeoJSON"
)

# I save a CSV version for my statistical analysis.
# I keep the coordinates but remove the geometry column.

battery_tx_plants.to_csv(
    battery_output_path + "Texas_Battery_Plants.csv",
    index=False
)

# I confirm that the files have been saved.

print("Battery GeoJSON saved successfully.")
print("Battery CSV saved successfully.")
print("Total operating battery plants:", len(battery_tx_map))
print("Total operating battery capacity (MW):", battery_tx_map["capacity_mw"].sum())

Battery GeoJSON saved successfully.
Battery CSV saved successfully.
Total operating battery plants: 200
Total operating battery capacity (MW): 13909.3


### 1.5. loading oil and gas wells using Texas Railroad Commission (RRC) data.

In [26]:
import os

# I define the folder containing my downloaded
# Railroad Commission oil and gas well data.

wells_path = raw_data_path + "RRC_Wells/"

# I inspect the files in this folder to identify
# the Archer County shapefile components.
# I start with one county so I can understand
# the data structure before processing Texas statewide.

print("Files in RRC_Wells:")
for item in os.listdir(wells_path):
    print(item)

wells_extracted_path = wells_path + "documents_20261015/"

Files in RRC_Wells:
documents_20261015.zip
documents_20261015
documents_20261016.zip
documents_20261016


In [27]:
import zipfile

# I define the path to the Archer County well dataset.Archer is a pilot

archer_wells_path = wells_extracted_path + "well009.zip"

# I inspect the ZIP file to identify its contents.
# A shapefile usually includes .shp, .shx, .dbf,
# and .prj files that work together.

with zipfile.ZipFile(archer_wells_path, "r") as zip_file:
    print("Files inside Archer County ZIP:")
    for filename in zip_file.namelist():
        print(filename)

Files inside Archer County ZIP:
well009b.cpg
well009b.dbf
well009b.prj
well009b.sbn
well009b.sbx
well009b.shp
well009b.shp.xml
well009b.shx
well009l.cpg
well009l.dbf
well009l.prj
well009l.sbn
well009l.sbx
well009l.shp
well009l.shp.xml
well009l.shx
well009s.cpg
well009s.dbf
well009s.prj
well009s.sbn
well009s.sbx
well009s.shp
well009s.shp.xml
well009s.shx


In [28]:
import geopandas as gpd

# I load the three geographic layers directly from
# the Archer County ZIP file without extracting it.

archer_b = gpd.read_file(
    "zip://" + archer_wells_path + "!well009b.shp"
)

archer_l = gpd.read_file(
    "zip://" + archer_wells_path + "!well009l.shp"
)

archer_s = gpd.read_file(
    "zip://" + archer_wells_path + "!well009s.shp"
)

# I compare the number of records, geometry types,
# and coordinate systems across the three layers.

for name, data in [
    ("well009b", archer_b),
    ("well009l", archer_l),
    ("well009s", archer_s)
]:
    print("\nLayer:", name)
    print("Records:", len(data))
    print("Geometry types:")
    print(data.geometry.geom_type.value_counts())
    print("CRS:", data.crs)
    print("Columns:", data.columns.tolist())


Layer: well009b
Records: 40192
Geometry types:
Point    40192
Name: count, dtype: int64
CRS: EPSG:4267
Columns: ['BOTTOM_ID', 'SURFACE_ID', 'SYMNUM', 'APINUM', 'RELIAB', 'API10', 'API', 'LONG27', 'LAT27', 'LONG83', 'LAT83', 'OUT_FIPS', 'CWELLNUM', 'RADIOACT', 'WELLID', 'STCODE', 'geometry']

Layer: well009l
Records: 14
Geometry types:
LineString    14
Name: count, dtype: int64
CRS: EPSG:4267
Columns: ['BOTTOM_ID', 'SURFACE_ID', 'API10', 'API', 'STCODE', 'SHAPE_LEN', 'geometry']

Layer: well009s
Records: 40188
Geometry types:
Point    40188
Name: count, dtype: int64
CRS: EPSG:4267
Columns: ['SURFACE_ID', 'SYMNUM', 'API', 'RELIAB', 'LONG27', 'LAT27', 'LONG83', 'LAT83', 'WELLID', 'geometry']


In [29]:
### Inspect well identifiers and attributes

In [30]:
# I inspect the first five surface well records
# to understand how the Railroad Commission
# identifies and describes well locations.

display(archer_s.head())

# I check whether the well identifiers are unique.
# I want to know whether each row represents
# a separate well or whether some records repeat.

print("Total records:", len(archer_s))

for column in ["SURFACE_ID", "API", "WELLID"]:
    print("\nColumn:", column)
    print("Unique values:", archer_s[column].nunique())
    print("Missing values:", archer_s[column].isna().sum())
    print("Duplicate values:", archer_s[column].duplicated().sum())

# I inspect the available symbol and reliability codes.
# These may help me understand how well locations
# are classified in the RRC geographic dataset.

print("\nSYMNUM values:")
print(archer_s["SYMNUM"].value_counts(dropna=False).head(20))

print("\nRELIAB values:")
print(archer_s["RELIAB"].value_counts(dropna=False))

,SURFACE_ID,SYMNUM,API,RELIAB,LONG27,LAT27,LONG83,LAT83,WELLID,geometry
0,922361,7,00920186,15,-98.732472,33.780651,-98.732818,33.780764,20186,POINT (-98.73247 33.78065)
1,922964,4,00930465,15,-98.667804,33.754106,-98.668148,33.754220,30465,POINT (-98.6678 33.75411)
2,880176,7,00935729,15,-98.640849,33.414828,-98.640849,33.414828,35729,POINT (-98.64085 33.41483)
3,1384462,19,00943222,55,-98.760020,33.610152,-98.760366,33.610269,43222,POINT (-98.76002 33.61015)
4,861883,4,00983359,15,-98.915590,33.461046,-98.915590,33.461046,83359,POINT (-98.91559 33.46105)


Total records: 40188

Column: SURFACE_ID
Unique values: 40188
Missing values: 0
Duplicate values: 0

Column: API
Unique values: 18219
Missing values: 0
Duplicate values: 21969

Column: WELLID
Unique values: 18159
Missing values: 21969
Duplicate values: 22028

SYMNUM values:
SYMNUM
3     13990
7     12404
4      7378
2      4072
9      1574
21      381
11      237
19       66
10       20
74       18
86       12
6        10
75        8
8         7
5         6
87        2
17        1
23        1
20        1
Name: count, dtype: int64

RELIAB values:
RELIAB
15    33309
40     3752
30     1219
55      691
20      410
50      395
60      333
45       48
25       10
10        9
16        7
48        3
17        2
Name: count, dtype: int64


the American Petroleum Institute, and an API well number is a unique, permanent identification number assigned to every oil and gas well drilled in the United States

In [31]:
## we have 40,188 unique surface records, 
#but that doesn't necessarily mean 40,188 unique oil and gas wells.

In [32]:
## Examine repeated API values

In [33]:
# I examine the API values that appear multiple times.
# I want to determine whether these are genuine duplicate
# wells or different records associated with the same API.

api_counts = archer_s["API"].value_counts()

print("Most frequently repeated API values:")
print(api_counts.head(10))

# I select one repeated API value and inspect its records.
# This helps me understand whether the same API is
# associated with different surface locations.

example_api = api_counts.index[0]

print("\nExample API:", example_api)

display(
    archer_s.loc[
        archer_s["API"] == example_api,
        ["SURFACE_ID", "SYMNUM", "API", "WELLID", "geometry"]
    ].head(15)
)

# I check how many records have an API value of zero
# or a missing API value, since these could explain
# why the identifier repeats so frequently.

print("\nAPI values equal to zero:",
      (archer_s["API"] == 0).sum())

print("Missing API values:",
      archer_s["API"].isna().sum())

Most frequently repeated API values:
API
009         21947
485            19
503             4
00904932        2
00981901        2
00983360        1
00938468        1
00937026        1
00933083        1
00986630        1
Name: count, dtype: int64

Example API: 009


,SURFACE_ID,SYMNUM,API,WELLID,geometry
27,921929,2,009,None,POINT (-98.66865 33.80679)
28,924741,7,009,None,POINT (-98.70193 33.80701)
32,934358,3,009,None,POINT (-98.47749 33.8315)
33,934284,3,009,None,POINT (-98.45696 33.8289)
35,934456,3,009,None,POINT (-98.48323 33.82799)
36,934458,3,009,None,POINT (-98.49896 33.82789)
37,934359,3,009,None,POINT (-98.48708 33.82627)
39,934422,7,009,None,POINT (-98.47732 33.82439)
41,934423,7,009,None,POINT (-98.47717 33.82282)
43,934263,2,009,None,POINT (-98.44828 33.82168)



API values equal to zero: 0
Missing API values: 0


I've found the main reason for the repeated API values.
21,947 surface records have API = "009", which is Archer County's three-digit county code, rather than a complete well identifier. These records also appear to have missing WELLID values.
This changes how I should approach the analysis.
I shouldn't remove these records simply because their API numbers are incomplete. They may still represent real, mapped well locations. However, I also shouldn't treat them as independently verified, uniquely identified wells.
For now, I keep all 40,188 surface records and distinguishing between records with complete and incomplete API identifiers.

In [34]:
## Check API completeness and repeated locations

In [35]:
# I examine the length of each API value.
# I want to distinguish complete identifiers from
# shorter values that may only represent county codes.

archer_s["api_length"] = (
    archer_s["API"].astype("string").str.strip().str.len()
)

print("API identifier lengths:")
print(archer_s["api_length"].value_counts(dropna=False).sort_index())

# I check whether different surface records share
# exactly the same geographic coordinates.
# A repeated coordinate does not automatically mean
# a duplicate well, but it deserves closer inspection.

print("\nUnique surface coordinates:",
      archer_s.geometry.nunique())

print("Records with repeated coordinates:",
      archer_s.geometry.duplicated(keep=False).sum())

# I examine the records associated with repeated coordinates.

repeated_locations = archer_s[
    archer_s.geometry.duplicated(keep=False)
].copy()

display(
    repeated_locations[
        ["SURFACE_ID", "SYMNUM", "API", "WELLID", "geometry"]
    ].head(15)
)

API identifier lengths:
api_length
3    21971
6        1
8    18216
Name: count, dtype: Int64

Unique surface coordinates: 40176
Records with repeated coordinates: 23


,SURFACE_ID,SYMNUM,API,WELLID,geometry
4886,1201471,2,00942092,42092,POINT (-98.57586 33.7372)
7432,899384,9,00933695,33695,POINT (-98.76032 33.68568)
10168,884155,4,009,None,POINT (-98.54931 33.6232)
10169,884154,4,009,None,POINT (-98.54931 33.6232)
10176,884152,9,009,None,POINT (-98.55746 33.62249)
12625,883736,3,009,None,POINT (-98.56431 33.51305)
14124,895685,7,00930063,30063,POINT (-98.85362 33.52768)
14125,895686,4,009,None,POINT (-98.85362 33.52768)
17418,1297544,3,00942766,42766,POINT (-98.76452 33.46674)
17419,1079708,2,00941371,41371,POINT (-98.76452 33.46674)


The 23 records with repeated coordinates represent a very small portion of the dataset. Some have different API numbers, so I should not automatically delete them. Multiple wells can share a surface location.
The bigger concern is that approximately 55% of the surface records have only a three-character API value. These appear to be county codes rather than complete identifiers.
Before cleaning or filtering anything, I want to understand what the RRC's SYMNUM field represents. It might distinguish different types of wells.

In [36]:
# I compare the RRC symbol codes with API identifier lengths.
# I want to see whether records with incomplete API values
# are concentrated in particular symbol categories.

symbol_api_summary = pd.crosstab(
    archer_s["SYMNUM"],
    archer_s["api_length"]
)

# I add the total number of records for each symbol code.
symbol_api_summary["total"] = symbol_api_summary.sum(axis=1)

# I calculate the percentage of records with an
# 8-character API identifier within each symbol category.

symbol_api_summary["percent_8_digit_api"] = (
    symbol_api_summary.get(8, 0)
    / symbol_api_summary["total"]
    * 100
).round(1)

# I sort the results by the most common symbol codes.

symbol_api_summary = symbol_api_summary.sort_values(
    "total",
    ascending=False
)

display(symbol_api_summary)

api_length,3,6,8,total,percent_8_digit_api
SYMNUM,,,,,
3,10503,0,3487,13990,24.9
7,4453,0,7951,12404,64.1
4,4163,1,3214,7378,43.6
2,2527,0,1545,4072,37.9
9,205,0,1369,1574,87.0
21,16,0,365,381,95.8
11,85,0,152,237,64.1
19,0,0,66,66,100.0
10,3,0,17,20,85.0


This table shows something important: the completeness of API identifiers varies considerably across the RRC symbol categories.
For example, only 24.9% of records with SYMNUM = 3 have an eight-character API identifier, compared with 87% for SYMNUM = 9 and 95.8% for SYMNUM = 21.
This suggests that the symbol categories may represent different kinds of well records, but I still cannot determine their operating status from these numbers alone.
I keep two measures separate:
- Mapped well surface locations: All RRC surface records, regardless of whether an API identifier is complete.
- Operating oil and gas wells: Only wells whose operating status can be established from reliable RRC data.

I found the official RRC symbol definitions.here: https://www.rrc.texas.gov/media/kmld3uzj/digital-map-information-user-guide.pdf (page 18), many mapped RRC records represent dry holes, plugged wells, or canceled locations, rather than currently operating production wells.

In [37]:
# I use the RRC Digital Map Information User Guide
# to assign descriptions to the well symbol codes.

well_symbols = {
    2: "Permitted location",
    3: "Dry hole",
    4: "Oil well",
    5: "Gas well",
    6: "Oil/gas well",
    7: "Plugged oil well",
    8: "Plugged gas well",
    9: "Canceled location",
    10: "Plugged oil/gas well",
    11: "Injection/disposal well",
    19: "Shut-in oil well",
    20: "Shut-in gas well",
    21: "Injection/disposal from oil",
    22: "Injection/disposal from gas",
    23: "Injection/disposal from oil/gas",
    74: "Water supply well",
    75: "Water supply from oil",
    86: "Horizontal well surface location",
    87: "Directional/sidetrack well surface location"
}

# I preserve the original SYMNUM field and add
# a readable classification for each surface record.

archer_s["well_class"] = archer_s["SYMNUM"].map(well_symbols)

# I check the classifications and identify any codes
# that still need definitions.

print("Well classifications:")
print(archer_s["well_class"].value_counts(dropna=False))

print("\nCodes without descriptions:")
print(
    archer_s.loc[
        archer_s["well_class"].isna(), "SYMNUM"
    ].value_counts()
)

Well classifications:
well_class
Dry hole                                       13990
Plugged oil well                               12404
Oil well                                        7378
Permitted location                              4072
Canceled location                               1574
Injection/disposal from oil                      381
Injection/disposal well                          237
Shut-in oil well                                  66
Plugged oil/gas well                              20
Water supply well                                 18
Horizontal well surface location                  12
Oil/gas well                                      10
Water supply from oil                              8
Plugged gas well                                   7
Gas well                                           6
Directional/sidetrack well surface location        2
NaN                                                1
Injection/disposal from oil/gas                    1
Shut-in gas w

In [38]:
# I inspect the one record that does not have
# a classification in my symbol dictionary.
# I keep it in the dataset rather than excluding it.

display(
    archer_s.loc[
        archer_s["SYMNUM"] == 17,
        ["SURFACE_ID", "SYMNUM", "API", "WELLID", "geometry"]
    ]
)

# I summarize the main groups of mapped well records.
# These groups describe map classifications, not
# independently verified current operating status.

production_symbols = [4, 5, 6]
plugged_symbols = [7, 8, 10]
dry_hole_symbols = [3]
permitted_symbols = [2]
canceled_symbols = [9]

print("Oil/gas well symbols:",
      archer_s["SYMNUM"].isin(production_symbols).sum())

print("Plugged well symbols:",
      archer_s["SYMNUM"].isin(plugged_symbols).sum())

print("Dry holes:",
      archer_s["SYMNUM"].isin(dry_hole_symbols).sum())

print("Permitted locations:",
      archer_s["SYMNUM"].isin(permitted_symbols).sum())

print("Canceled locations:",
      archer_s["SYMNUM"].isin(canceled_symbols).sum())

print("Other classifications:",
      (~archer_s["SYMNUM"].isin(
          production_symbols
          + plugged_symbols
          + dry_hole_symbols
          + permitted_symbols
          + canceled_symbols
      )).sum())

,SURFACE_ID,SYMNUM,API,WELLID,geometry
538,921843,17,009,None,POINT (-98.69365 33.81025)


Oil/gas well symbols: 7394
Plugged well symbols: 12431
Dry holes: 13990
Permitted locations: 4072
Canceled locations: 1574
Other classifications: 727


In [39]:
#The category field will be useful for our interactive map because 
#we can allow users to distinguish production-type wells from historical or other well infrastructure.
# I create broader well categories 
# I distinguish production-type well symbols from
# plugged wells, dry holes, and permitted locations.
#note to my self: I do not assume production-type symbols prove
# that a well is currently operating.

well_categories = {
    2: "Permitted location",
    3: "Dry hole",
    4: "Oil/gas well",
    5: "Oil/gas well",
    6: "Oil/gas well",
    7: "Plugged well",
    8: "Plugged well",
    9: "Canceled location",
    10: "Plugged well",
    11: "Other well infrastructure",
    19: "Shut-in well",
    20: "Shut-in well",
    21: "Other well infrastructure",
    22: "Other well infrastructure",
    23: "Other well infrastructure",
    74: "Other well infrastructure",
    75: "Other well infrastructure",
    86: "Other well infrastructure",
    87: "Other well infrastructure"
}

# I preserve the original RRC symbol codes and
# add a broader category for each mapped record.
# Any undefined code remains classified as Unknown.

archer_s["well_category"] = (
    archer_s["SYMNUM"]
    .map(well_categories)
    .fillna("Unknown")
)

# I check that every surface record is assigned
# to exactly one category.

print("Total records:", len(archer_s))

print("\nRecords by category:")
print(archer_s["well_category"].value_counts())

print("\nMissing categories:",
      archer_s["well_category"].isna().sum())

Total records: 40188

Records by category:
well_category
Dry hole                     13990
Plugged well                 12431
Oil/gas well                  7394
Permitted location            4072
Canceled location             1574
Other well infrastructure      659
Shut-in well                    67
Unknown                          1
Name: count, dtype: int64

Missing categories: 0


In [40]:
import glob

# I use glob to find files whose names match a specific pattern.
# I expect 254 ZIP files because Texas has 254 counties.

# I search for files named wellXXX.zip, where XXX is a three-digit code.
# Each [0-9] represents exactly one digit between 0 and 9.
# For example, this pattern matches well001.zip and well254.zip.
# I use sorted() to organize the matching filenames alphabetically.

well_zip_files = sorted(
    glob.glob(wells_extracted_path + "well[0-9][0-9][0-9].zip")
)

# I check how many county ZIP files were found.

print("Total county ZIP files:", len(well_zip_files))


# I check whether each ZIP file contains the surface-location
# shapefile needed for my spatial analysis.
# I create an empty list to record counties with missing shapefiles.

missing_surface_files = []

# I examine each county ZIP file individually.

for path in well_zip_files:

    # I extract the three-digit county code from the ZIP filename.
    # For example, in "well001.zip", [4:7] extracts "001".
    # Python starts counting positions at 0:
    # w=0, e=1, l=2, l=3, 0=4, 0=5, 1=6.
    # The slice [4:7] includes positions 4, 5, and 6,
    # but excludes position 7.

    county_code = os.path.basename(path)[4:7]

    # I open the ZIP file without extracting its contents.
    # namelist() returns the names of all files inside the ZIP.
    # basename() removes any folder paths from those filenames.
    # lower() makes the filenames lowercase for consistent comparison.

    with zipfile.ZipFile(path) as zip_file:
        filenames = [
            os.path.basename(name).lower()
            for name in zip_file.namelist()
        ]

    # I check whether the expected surface shapefile exists.
    # For example, well001.zip should contain well001s.shp.
    # If it is missing, I add the county code to my list.

    if f"well{county_code}s.shp" not in filenames:
        missing_surface_files.append(county_code)

# I display counties missing the required surface shapefile.
# An empty list means every ZIP file found contains its expected .shp file.

print("Counties missing surface shapefiles:", missing_surface_files)
# I stop if the source inventory is incomplete, because missing counties bias statewide counts.
if len(well_zip_files) != 254 or missing_surface_files:
    raise ValueError("I need all 254 well ZIP files and their surface shapefiles before continuing.")


Total county ZIP files: 254
Counties missing surface shapefiles: []


In [41]:
# I compare five county surface shapefiles to confirm
# that their columns and coordinate systems are consistent.

for i in [0, 50, 100, 150, 253]:
    path = well_zip_files[i]
    county_code = os.path.basename(path)[4:7]

    sample = gpd.read_file(
        "zip://" + path + f"!well{county_code}s.shp",
        rows=5
    )

    print("\nCounty code:", county_code)
    print("CRS:", sample.crs)
    print("Columns:", sample.columns.tolist())


County code: 001
CRS: EPSG:4267
Columns: ['SURFACE_ID', 'SYMNUM', 'API', 'RELIAB', 'LONG27', 'LAT27', 'LONG83', 'LAT83', 'WELLID', 'geometry']

County code: 101
CRS: EPSG:4267
Columns: ['SURFACE_ID', 'SYMNUM', 'API', 'RELIAB', 'LONG27', 'LAT27', 'LONG83', 'LAT83', 'WELLID', 'geometry']

County code: 201
CRS: EPSG:4267
Columns: ['SURFACE_ID', 'SYMNUM', 'API', 'RELIAB', 'LONG27', 'LAT27', 'LONG83', 'LAT83', 'WELLID', 'geometry']

County code: 301
CRS: EPSG:4267
Columns: ['SURFACE_ID', 'SYMNUM', 'API', 'RELIAB', 'LONG27', 'LAT27', 'LONG83', 'LAT83', 'WELLID', 'geometry']

County code: 507
CRS: EPSG:4267
Columns: ['SURFACE_ID', 'SYMNUM', 'API', 'RELIAB', 'LONG27', 'LAT27', 'LONG83', 'LAT83', 'WELLID', 'geometry']


In [42]:
#This cell may take several minutes ( more than 20 minutes, sorry data checker, but it's necessary)) and require substantial RAM because we're combining all statewide point geometries.
# I create a list to store the well records from each county.
# I read one county at a time rather than opening all
# 254 shapefiles simultaneously.

all_texas_wells = []

for i, path in enumerate(well_zip_files):
    county_code = os.path.basename(path)[4:7]

    # I load the surface well locations from each county ZIP.
    # I keep the original geographic coordinates and well identifiers.

    county_wells = gpd.read_file(
        "zip://" + path + f"!well{county_code}s.shp",
        columns=["SURFACE_ID", "SYMNUM", "API", "RELIAB", "WELLID"]
    )

    # I record the county code associated with the source file.
    # I preserve it as a string so leading zeros are not lost.

    county_wells["county_fips"] = county_code

    # I use my existing symbol dictionary to classify
    # each mapped well surface record.

    county_wells["well_category"] = (
        county_wells["SYMNUM"]
        .map(well_categories)
        .fillna("Unknown")
    )

    if county_wells.crs is None:
        raise ValueError(f"I cannot verify the CRS for county {county_code} wells.")
    county_wells = county_wells.to_crs("EPSG:4267")
    all_texas_wells.append(county_wells)

    # I print progress every 25 counties so I can
    # see how far the statewide processing has progressed.

    if (i + 1) % 25 == 0:
        print("Counties processed:", i + 1)

# I combine the county datasets into one GeoDataFrame.
# I preserve the EPSG:4267 coordinate system for now.

texas_wells = gpd.GeoDataFrame(
    pd.concat(all_texas_wells, ignore_index=True),
    crs="EPSG:4267"
)

print("\nTotal counties processed:", len(well_zip_files))
print("Total mapped surface records:", len(texas_wells))
print("Coordinate system:", texas_wells.crs)

Counties processed: 25
Counties processed: 50
Counties processed: 75
Counties processed: 100
Counties processed: 125
Counties processed: 150
Counties processed: 175
Counties processed: 200
Counties processed: 225
Counties processed: 250

Total counties processed: 254
Total mapped surface records: 1380352
Coordinate system: EPSG:4267


In [43]:
# I check whether the same surface identifier appears
# more than once in the combined statewide dataset.

print("Total surface records:", len(texas_wells))

print(
    "Unique surface IDs:",
    texas_wells["SURFACE_ID"].nunique()
)

print(
    "Duplicate surface IDs:",
    texas_wells["SURFACE_ID"].duplicated().sum()
)

# I check whether any mapped locations are missing geometry.

print(
    "\nMissing geometries:",
    texas_wells.geometry.isna().sum()
)

print(
    "Empty geometries:",
    texas_wells.geometry.is_empty.sum()
)

# I summarize the statewide records by well category.

print("\nStatewide well categories:")
print(texas_wells["well_category"].value_counts())

Total surface records: 1380352
Unique surface IDs: 1380349
Duplicate surface IDs: 3

Missing geometries: 0
Empty geometries: 0

Statewide well categories:
well_category
Plugged well                 409259
Oil/gas well                 336468
Dry hole                     288905
Other well infrastructure    212084
Permitted location            88272
Canceled location             39438
Shut-in well                   3711
Unknown                        2215
Name: count, dtype: int64


Note to my self:these are mapped surface records, not necessarily 1.38 million currently operating wells. The dataset includes historical, plugged, canceled, and other well locations.

In [44]:
# I identify all records with duplicated surface IDs.
# I keep both the first occurrence and its duplicates
# so I can compare their attributes and locations.

duplicate_wells = texas_wells[
    texas_wells["SURFACE_ID"].duplicated(keep=False)
].copy()

# I sort the records to make comparisons easier.

duplicate_wells = duplicate_wells.sort_values("SURFACE_ID")

print("Records with duplicated surface IDs:", len(duplicate_wells))

print(
    duplicate_wells[
        ["SURFACE_ID", "county_fips", "API",
         "SYMNUM", "well_category", "geometry"]
    ].to_string(index=False)
)

Records with duplicated surface IDs: 6
 SURFACE_ID county_fips      API  SYMNUM well_category                    geometry
         10         061 06130404       3      Dry hole  POINT (-97.43298 25.90318)
         10         393 39331958      10  Plugged well POINT (-100.59109 35.87147)
    1029321         065 06581830       4  Oil/gas well POINT (-101.39494 35.62455)
    1029321         233 06581830       4  Oil/gas well POINT (-101.39494 35.62455)
    1108056         485 48732480       4  Oil/gas well  POINT (-98.95246 34.03181)
    1108056         487 48732480       4  Oil/gas well  POINT (-98.95246 34.03181)


In [45]:
# I preserve all statewide well surface records until I can verify individual duplicates.
# I do not treat SURFACE_ID as a unique statewide key because the original investigation
# found that repeated IDs can refer to different wells.
texas_wells_clean = texas_wells.copy()

# I save possible duplicate identifiers for manual review instead of removing records
# based only on an API county prefix or a hard-coded list of IDs.
well_duplicate_review = texas_wells_clean.loc[
    texas_wells_clean.duplicated("SURFACE_ID", keep=False)
].copy()
print("Well records requiring identifier review:", len(well_duplicate_review))
print("Records removed automatically: 0")


Well records requiring identifier review: 6
Records removed automatically: 0


In [46]:
# SO I think SURFACE_ID is not a perfectly unique statewide identifier, as the two different wells with ID 10 show.

In [47]:
# I identify which RRC symbol codes are not yet
# included in my well classification dictionary.

unknown_wells = texas_wells_clean[
    texas_wells_clean["well_category"] == "Unknown"
]

print("Unknown records:", len(unknown_wells))

print("\nUnknown symbol codes:")
print(
    unknown_wells["SYMNUM"]
    .value_counts(dropna=False)
    .sort_index()
)

Unknown records: 2215

Unknown symbol codes:
SYMNUM
1        2
12     763
16       8
17      37
18      97
27       1
36       3
47       4
73      89
76      25
77      29
78     118
79     175
80      15
81     104
88     586
89      23
90       1
103     33
104      3
105      4
106      3
108      2
110      2
114      1
116     35
118      2
121      1
124      4
126      1
136     24
140      3
144      2
148      1
152     14
Name: count, dtype: int64


In [48]:
#I shouldn't assign them to oil/gas, plugged, or other categories based on their numbers alone.
#I found this:https://www.rrc.state.tx.us/media/kmld3uzj/digital-map-information-user-guide.pdf (page 17)
#For example, it says code 79 as Observation from Oil, code 81 as Observation from Oil/Gas, and code 88 as Storage. These are important distinctions for our environmental footprint.

In [49]:
# I add the RRC symbol descriptions I have verified
# in the official Digital Map Information User Guide.

additional_symbols = {
    79: "Observation from Oil",
    80: "Observation from Gas",
    81: "Observation from Oil/Gas",
    88: "Storage"
}

# I classify these records as other well infrastructure
# rather than treating them as producing oil or gas wells.

for code in additional_symbols:
    well_categories[code] = "Other well infrastructure"

# I update my cleaned statewide dataset using
# the expanded classification dictionary.

texas_wells_clean["well_category"] = (
    texas_wells_clean["SYMNUM"]
    .map(well_categories)
    .fillna("Unknown")
)

print("Remaining unknown records:",
      (texas_wells_clean["well_category"] == "Unknown").sum())

Remaining unknown records: 1335


In [50]:
# I identify the remaining RRC symbol codes
# that are not included in my classification dictionary.

remaining_unknown = texas_wells_clean[
    texas_wells_clean["well_category"] == "Unknown"
]

# I count the remaining records by symbol code
# and sort them from most common to least common.

print("Remaining unknown records:", len(remaining_unknown))

print("\nRemaining unknown symbol codes:")
print(
    remaining_unknown["SYMNUM"]
    .value_counts(dropna=False)
    .to_string()
)

Remaining unknown records: 1335

Remaining unknown symbol codes:
SYMNUM
12     763
78     118
18      97
73      89
17      37
116     35
103     33
77      29
76      25
136     24
89      23
152     14
16       8
47       4
124      4
105      4
140      3
36       3
106      3
104      3
108      2
1        2
144      2
118      2
110      2
90       1
121      1
126      1
114      1
148      1
27       1


In [51]:
#Classify the rest:https://www.rrc.texas.gov/media/kmld3uzj/digital-map-information-user-guide.pdf

In [52]:
# I add four symbol codes confirmed in the RRC guide.
# I keep these separate from producing oil/gas wells.

additional_categories = {
    12: "Core test",
    18: "Other well infrastructure",
    73: "Other well infrastructure",
    78: "Other well infrastructure"
}

# I update my existing classification dictionary.

well_categories.update(additional_categories)

# I refresh the categories in my cleaned statewide data.

texas_wells_clean["well_category"] = (
    texas_wells_clean["SYMNUM"]
    .map(well_categories)
    .fillna("Unknown")
)

# I check how many records still need classification.

print("Remaining unknown records:",
      (texas_wells_clean["well_category"] == "Unknown").sum())

print("\nUpdated category counts:")
print(texas_wells_clean["well_category"].value_counts())

Remaining unknown records: 268

Updated category counts:
well_category
Plugged well                 409259
Oil/gas well                 336468
Dry hole                     288905
Other well infrastructure    213268
Permitted location            88272
Canceled location             39438
Shut-in well                   3711
Core test                       763
Unknown                         268
Name: count, dtype: int64


In [53]:
# I identify the remaining symbol codes that are
# not included in my classification dictionary.

remaining_unknown = texas_wells_clean[
    texas_wells_clean["well_category"] == "Unknown"
]

# I count how many records have each unknown symbol.
# I also check which counties contain these records.

unknown_summary = (
    remaining_unknown
    .groupby("SYMNUM")
    .agg(
        records=("SURFACE_ID", "size"),
        counties=("county_fips", "nunique")
    )
    .sort_values("records", ascending=False)
)

print(unknown_summary.to_string())

print("\nTotal unknown records:", len(remaining_unknown))

        records  counties
SYMNUM                   
17           37        15
116          35        13
103          33        10
77           29        19
76           25        20
136          24        12
89           23        11
152          14        10
16            8         5
105           4         4
47            4         4
124           4         4
140           3         2
106           3         3
36            3         3
104           3         1
144           2         2
1             2         2
118           2         1
110           2         1
108           2         2
90            1         1
27            1         1
114           1         1
121           1         1
126           1         1
148           1         1

Total unknown records: 268


Great.the remaining 268 unknown records are spread across 27 RRC symbol codes. That's just 0.019% of our 1,380,350 statewide records.
I'll keep these records as Unknown for now and their original SYMNUM values helps to investigate them later.

In [54]:
### Saving well file

In [55]:
# CSV file is very large so it takes time 
# I define the folder where I save my cleaned datasets again.

cleaned_data_path = file_path + "03_Cleaned_Data/"
os.makedirs(cleaned_data_path, exist_ok=True)

# I convert the coordinates from NAD27 to WGS84 (EPSG:4326).
# I use WGS84 so the well locations are compatible with interactive maps.

texas_wells_clean = texas_wells_clean.to_crs("EPSG:4326")

# I save all mapped surface well records as a GeoPackage.
# I preserve the geometry, original symbol codes, and well categories.

texas_wells_clean.to_file(
    cleaned_data_path + "Texas_Well_Surface_Locations.gpkg",
    layer="texas_wells",
    driver="GPKG"
)

# I create a separate CSV version of the complete well dataset.
# I extract longitude (x) and latitude (y) from the WGS84 point geometries.
# I keep the original attributes and remove the geometry column.

texas_wells_csv = texas_wells_clean.copy()

texas_wells_csv["longitude"] = texas_wells_csv.geometry.x
texas_wells_csv["latitude"] = texas_wells_csv.geometry.y

texas_wells_csv = texas_wells_csv.drop(columns="geometry")

# I save the CSV for sharing and statistical analysis.

texas_wells_csv.to_csv(
    cleaned_data_path + "Texas_Well_Surface_Locations.csv",
    index=False
)

# I confirm that both files have been saved.

print("Texas well GeoPackage saved successfully.")
print("Texas well CSV saved successfully.")
print("Total well records:", len(texas_wells_clean))
print("CRS:", texas_wells_clean.crs)

Texas well GeoPackage saved successfully.
Texas well CSV saved successfully.
Total well records: 1380352
CRS: EPSG:4326


In [56]:
# I calculate the number of mapped surface records
# in each well category across Texas.

state_well_summary = (
    texas_wells_clean
    .groupby("well_category")
    .size()
    .reset_index(name="well_count")
    .sort_values("well_count", ascending=False)
)

# I calculate well counts by county and category.
# I retain county FIPS codes as three-character strings.

county_well_summary = (
    texas_wells_clean
    .groupby(["county_fips", "well_category"])
    .size()
    .reset_index(name="well_count")
)

# I save both summary tables for my report and maps.

state_well_summary.to_csv(
    cleaned_data_path + "Texas_Well_State_Summary.csv",
    index=False
)

county_well_summary.to_csv(
    cleaned_data_path + "Texas_Well_County_Summary.csv",
    index=False
)

print("Statewide summary:")
print(state_well_summary.to_string(index=False))

print("\nCounties represented:",
      county_well_summary["county_fips"].nunique())

print("\nTotal records in county summary:",
      county_well_summary["well_count"].sum())

Statewide summary:
            well_category  well_count
             Plugged well      409259
             Oil/gas well      336468
                 Dry hole      288905
Other well infrastructure      213268
       Permitted location       88272
        Canceled location       39438
             Shut-in well        3711
                Core test         763
                  Unknown         268

Counties represented: 254

Total records in county summary: 1380352


these records represent mapped well surface locations, not the acreage physically occupied by oil and gas infrastructure. I'll need a separate method to estimate land footprint.

In [57]:
# I calculate the total number of mapped well surface
# records in each Texas county.

county_well_totals = (
    texas_wells_clean
    .groupby("county_fips")
    .size()
    .reset_index(name="total_well_records")
    .sort_values("total_well_records", ascending=False)
)

# I check the 15 counties with the most mapped records.

print("Top 15 counties:")
print(county_well_totals.head(15).to_string(index=False))

# I check that my county totals match the statewide total.

print("\nCounties:", len(county_well_totals))
print("Statewide records:", county_well_totals["total_well_records"].sum())

Top 15 counties:
county_fips  total_well_records
        485               46995
        009               40188
        503               30218
        003               27719
        135               24093
        417               22926
        329               20530
        479               20278
        237               18382
        371               18228
        461               18191
        401               17739
        227               17611
        317               17551
        059               17492

Counties: 254
Statewide records: 1380352


In [58]:
# I select records classified as oil/gas wells. and do not assume all these wells are currently operating.

oil_gas_wells = texas_wells_clean[
    texas_wells_clean["well_category"] == "Oil/gas well"
].copy()

# I count oil/gas well records in each county.

county_oil_gas_counts = (
    oil_gas_wells
    .groupby("county_fips")
    .size()
    .reset_index(name="oil_gas_well_records")
)

# I add the oil/gas counts to my county totals.
# I use zero for counties without records in this category.

county_well_totals = county_well_totals.merge(
    county_oil_gas_counts,
    on="county_fips",
    how="left"
)

county_well_totals["oil_gas_well_records"] = (
    county_well_totals["oil_gas_well_records"]
    .fillna(0)
    .astype(int)
)

# I save the county totals for my report.

county_well_totals.to_csv(
    cleaned_data_path + "Texas_Well_County_Totals.csv",
    index=False
)

print("Top 15 counties by oil/gas well records:")
print(
    county_well_totals
    .sort_values("oil_gas_well_records", ascending=False)
    .head(15)
    .to_string(index=False)
)

print("\nStatewide oil/gas well records:",
      county_well_totals["oil_gas_well_records"].sum())

Top 15 counties by oil/gas well records:
county_fips  total_well_records  oil_gas_well_records
        485               46995                 15469
        003               27719                 10805
        135               24093                  9053
        105               16413                  7895
        009               40188                  7394
        435                9167                  6329
        233               12897                  5895
        503               30218                  5886
        103               14601                  5559
        227               17611                  5028
        365               10667                  5005
        479               20278                  4963
        371               18228                  4937
        417               22926                  4919
        461               18191                  4732

Statewide oil/gas well records: 336468


### 1.6. Loading pipeline

In [59]:
# I define the folder containing the original RRC pipeline files.

pipelines_extracted_path = os.path.join(
    raw_data_path,
    "RRC_Pipelines",
    "documents_20261015"
)

# I find all county pipeline ZIP files.
# Each filename contains a three-digit county code.
# For example, pipeline001.zip represents county code 001.

pipeline_zip_files = sorted(
    glob.glob(
        os.path.join(
            pipelines_extracted_path,
            "pipeline[0-9][0-9][0-9].zip"
        )
    )
)

# I check the number of ZIP files before combining them.
# I expect 254 files because Texas has 254 counties.

print("Total pipeline ZIP files:", len(pipeline_zip_files))

# I inspect the first few filenames to confirm the pattern.

print("First five pipeline files:")
for path in pipeline_zip_files[:5]:
    print(os.path.basename(path))

Total pipeline ZIP files: 254
First five pipeline files:
pipeline001.zip
pipeline003.zip
pipeline005.zip
pipeline007.zip
pipeline009.zip


In [60]:
import zipfile

# I check that every county ZIP contains the pipeline
# line shapefile needed for my spatial analysis.

missing_pipeline_shapefiles = []

for path in pipeline_zip_files:

    # I extract the three-digit county code from the filename.
    # For example, pipeline001.zip gives "001".
    # The slice [8:11] selects positions 8, 9, and 10.

    county_code = os.path.basename(path)[8:11]

    # I inspect the ZIP contents without extracting the files.
    # I use basename() to remove folder paths and lower()
    # to make the filename comparison case-insensitive.

    with zipfile.ZipFile(path) as zip_file:
        filenames = [
            os.path.basename(name).lower()
            for name in zip_file.namelist()
        ]

    # I check for the expected shapefile components.
    # For example, county 001 should contain pipe001l.shp,
    # pipe001l.shx, and pipe001l.dbf.

    required_files = [
        f"pipe{county_code}l.shp",
        f"pipe{county_code}l.shx",
        f"pipe{county_code}l.dbf"
    ]

    missing_files = [
        name for name in required_files
        if name not in filenames
    ]

    if missing_files:
        missing_pipeline_shapefiles.append(
            (county_code, missing_files)
        )

# I report any missing components before loading the data.

print("Pipeline ZIP files checked:", len(pipeline_zip_files))
print("Counties with missing shapefile components:",
      len(missing_pipeline_shapefiles))

print("Missing components:", missing_pipeline_shapefiles)
# I require complete county coverage before I calculate statewide pipeline mileage.
if len(pipeline_zip_files) != 254:
    raise ValueError("I need all 254 county pipeline ZIP files.")


Pipeline ZIP files checked: 254
Counties with missing shapefile components: 0
Missing components: []


In [61]:
#This may take some time because Python must read and process 254 county shapefiles.
# I ran this code block by block before and combine them all here
# I create a list to collect pipeline records
# from all 254 Texas county shapefiles.

all_texas_pipelines = []

# I load each county pipeline shapefile separately.
# enumerate() gives me both the file position and its path.

for i, path in enumerate(pipeline_zip_files):

    # I extract the three-digit county code from the ZIP filename.
    # For example, pipeline001.zip gives "001".

    county_code = os.path.basename(path)[8:11]

    # I read the pipeline shapefile directly from the ZIP.
    # I select the attributes needed for my analysis
    # to avoid loading unnecessary columns.

    county_pipelines = gpd.read_file(
        "zip://" + path + f"!pipe{county_code}l.shp",
        columns=[
            "TPMS_ID", "OPER_NM", "SYS_NM",
            "DIAMETER", "CMDTY_DESC", "STATUS_CD",
            "SYSTYPE", "COUNTY", "ALBERS_MIL"
        ]
    )

    # I preserve the county code from the source ZIP.
    # I keep it separate from the original RRC COUNTY field.

    county_pipelines["source_county"] = county_code

    # I project the pipeline geometries to EPSG:3083.
    # This CRS uses meters and is suitable for measuring
    # pipeline lengths across Texas.

    if county_pipelines.crs is None:
        raise ValueError(f"I cannot verify the CRS for county {county_code} pipelines.")
    county_projected = county_pipelines.to_crs("EPSG:3083")

    # I calculate the mapped length of each pipeline segment.
    # I divide meters by 1609.344 to convert to miles.
    # I calculate lengths from geometries because ALBERS_MIL
    # may repeat across county files for the same pipeline ID.

    county_projected["geometry_miles"] = (
        county_projected.geometry.length / 1609.344
    )

    # I convert the geometries to WGS84 (EPSG:4326)
    # so they are compatible with my other cleaned datasets.

    county_projected = county_projected.to_crs("EPSG:4326")

    # I add this county's records to my statewide list.

    all_texas_pipelines.append(county_projected)

    # I print progress after every 25 county files.

    if (i + 1) % 25 == 0:
        print("Counties processed:", i + 1)

# I combine all county records into one GeoDataFrame.
# ignore_index=True creates a continuous row index.

texas_pipelines = gpd.GeoDataFrame(
    pd.concat(all_texas_pipelines, ignore_index=True),
    crs="EPSG:4326"
)

# I check the total records and mapped mileage
# before performing any duplicate removal.

print("\nCounty files processed:", len(pipeline_zip_files))
print("Total pipeline records:", len(texas_pipelines))
print(
    "Total mapped pipeline miles:",
    round(texas_pipelines["geometry_miles"].sum(), 2)
)

Counties processed: 25
Counties processed: 50
Counties processed: 75
Counties processed: 100
Counties processed: 125
Counties processed: 150
Counties processed: 175
Counties processed: 200
Counties processed: 225
Counties processed: 250

County files processed: 254
Total pipeline records: 496324
Total mapped pipeline miles: 363844.21


In [62]:
## Investigate duplicate pipeline records

In [63]:
# I investigate repeated pipeline identifiers before removing any records.
# I know that the same TPMS_ID may represent multiple pipeline segments,
# so repeated identifiers do not necessarily indicate duplicate records.

print("Total pipeline records:", len(texas_pipelines))

print(
    "Unique TPMS_ID values:",
    texas_pipelines["TPMS_ID"].nunique()
)

print(
    "Records with repeated TPMS_ID:",
    texas_pipelines["TPMS_ID"].duplicated(keep=False).sum()
)

# I identify records with identical TPMS_ID and geometry.
# I convert geometry to WKB, which represents geometry as binary data.
# This allows me to compare geometries without changing their coordinates.

texas_pipelines["geometry_wkb"] = (
    texas_pipelines.geometry.to_wkb()
)

# I check whether the same pipeline ID and identical geometry
# appear more than once in the combined statewide dataset.

duplicate_pipeline_records = texas_pipelines.duplicated(
    subset=["TPMS_ID", "geometry_wkb"],
    keep=False
)

print(
    "Records with identical TPMS_ID and geometry:",
    duplicate_pipeline_records.sum()
)

# I inspect a sample of potentially duplicated records.
# I retain the source county to understand where duplicates originate.

display(
    texas_pipelines.loc[
        duplicate_pipeline_records,
        [
            "TPMS_ID",
            "source_county",
            "STATUS_CD",
            "geometry_miles"
        ]
    ].head(10)
)

Total pipeline records: 496324
Unique TPMS_ID values: 489043
Records with repeated TPMS_ID: 14406
Records with identical TPMS_ID and geometry: 38


,TPMS_ID,source_county,STATUS_CD,geometry_miles
14818,2347964,007,B,0.002918
14819,2347334,007,B,0.002918
14820,2347336,007,B,0.002918
14821,2347338,007,B,0.002918
14981,2347642,007,B,0.002918
15004,2347322,007,B,0.002918
15005,2347324,007,B,0.002918
15006,2347962,007,B,0.002918
15007,2347966,007,B,0.002918
15008,2347326,007,B,0.002918


In [64]:
# I investigate records with identical TPMS_ID and geometry.
# I want to understand the duplicates before removing them.

duplicate_groups = texas_pipelines.loc[
    duplicate_pipeline_records
].copy()

# I count how many records belong to each duplicate group.
# I use dropna=False so missing identifiers are included.

duplicate_group_sizes = (
    duplicate_groups
    .groupby(["TPMS_ID", "geometry_wkb"], dropna=False)
    .size()
)

# I calculate how many records would be removed
# if I kept one record from each duplicate group.

records_to_remove = (
    duplicate_group_sizes - 1
).sum()

print("Records in duplicate groups:", len(duplicate_groups))
print("Number of duplicate groups:", len(duplicate_group_sizes))
print("Records that would be removed:", records_to_remove)

# I check whether duplicate records have different status codes.
# Different statuses could indicate conflicting information
# that I need to investigate before removing records.

status_conflicts = (
    duplicate_groups
    .groupby(["TPMS_ID", "geometry_wkb"], dropna=False)["STATUS_CD"]
    .nunique(dropna=False)
)

print(
    "Duplicate groups with conflicting statuses:",
    (status_conflicts > 1).sum()
)

# I calculate the mileage associated with redundant records.
# I do not remove anything yet.

redundant_records = texas_pipelines.duplicated(
    subset=["TPMS_ID", "geometry_wkb"],
    keep="first"
)

print(
    "Potential duplicate mileage:",
    round(
        texas_pipelines.loc[
            redundant_records, "geometry_miles"
        ].sum(),
        4
    )
)

# I display the potential duplicates together.
# I sort by TPMS_ID to make repeated records easier to compare.

display(
    duplicate_groups[
        [
            "TPMS_ID",
            "source_county",
            "STATUS_CD",
            "geometry_miles"
        ]
    ].sort_values("TPMS_ID").head(20)
)

Records in duplicate groups: 38
Number of duplicate groups: 19
Records that would be removed: 19
Duplicate groups with conflicting statuses: 0
Potential duplicate mileage: 0.1033


,TPMS_ID,source_county,STATUS_CD,geometry_miles
15004,2347322,007,B,0.002918
374073,2347322,391,B,0.002918
374074,2347324,391,B,0.002918
15005,2347324,007,B,0.002918
15008,2347326,007,B,0.002918
374077,2347326,391,B,0.002918
15031,2347328,007,B,0.002918
374241,2347328,391,B,0.002918
374149,2347330,391,B,0.002918
15021,2347330,007,B,0.002918


In [65]:
# I remove only exact repeated pipeline records, including their classification attributes.
# I do not collapse records merely because TPMS_ID and geometry match: a status,
# system type, or operator conflict requires review rather than an arbitrary choice.
pipeline_identity_columns = [
    "TPMS_ID", "geometry_wkb", "OPER_NM", "SYS_NM", "DIAMETER",
    "CMDTY_DESC", "STATUS_CD", "SYSTYPE", "COUNTY", "ALBERS_MIL"
]
exact_pipeline_duplicates = texas_pipelines.duplicated(
    subset=pipeline_identity_columns, keep="first"
)
pipeline_duplicates_removed = texas_pipelines.loc[exact_pipeline_duplicates].copy()
texas_pipelines_clean = texas_pipelines.loc[~exact_pipeline_duplicates].copy()
texas_pipelines_clean = texas_pipelines_clean.drop(columns="geometry_wkb")
print("Original pipeline records:", len(texas_pipelines))
print("Exact duplicate records removed:", len(pipeline_duplicates_removed))
print("Cleaned pipeline records:", len(texas_pipelines_clean))
print("Cleaned pipeline miles:", texas_pipelines_clean["geometry_miles"].sum())
assert len(texas_pipelines_clean) + len(pipeline_duplicates_removed) == len(texas_pipelines)

Original pipeline records: 496324
Exact duplicate records removed: 19
Cleaned pipeline records: 496305
Cleaned pipeline miles: 363844.1082118045


In [66]:
# Investigate pipeline status codes

In [67]:
# I examine the pipeline status codes before filtering the dataset.
# I want to understand how many records belong to each status
# and how much mapped pipeline mileage each status represents.

pipeline_status_summary = (
    texas_pipelines_clean
    .groupby("STATUS_CD", dropna=False)
    .agg(
        pipeline_records=("TPMS_ID", "size"),
        pipeline_miles=("geometry_miles", "sum")
    )
    .reset_index()
)

# I calculate the percentage of total mapped pipeline mileage
# represented by each status category.

pipeline_status_summary["percent_miles"] = (
    pipeline_status_summary["pipeline_miles"]
    / texas_pipelines_clean["geometry_miles"].sum()
    * 100
)

# I sort the results by pipeline mileage
# to identify the most common status categories.

pipeline_status_summary = pipeline_status_summary.sort_values(
    "pipeline_miles",
    ascending=False
)

display(pipeline_status_summary.round(2))


# I also inspect the pipeline system types.
# I want to understand which types of pipeline infrastructure
# are included before making decisions about the final dataset.

pipeline_system_summary = (
    texas_pipelines_clean
    .groupby("SYSTYPE", dropna=False)
    .agg(
        pipeline_records=("TPMS_ID", "size"),
        pipeline_miles=("geometry_miles", "sum")
    )
    .reset_index()
    .sort_values("pipeline_miles", ascending=False)
)

display(pipeline_system_summary.round(2))

,STATUS_CD,pipeline_records,pipeline_miles,percent_miles
1,I,435979,315158.02,86.62
0,B,59017,47362.57,13.02
2,R,1309,1323.52,0.36


,SYSTYPE,pipeline_records,pipeline_miles
2,G,373489,193339.58
8,T,58097,55778.67
7,Q,19584,38887.77
5,O,6977,32249.69
4,L,33213,27645.81
6,P,3834,12901.17
3,K,403,2052.11
9,Z,585,697.06
1,C,39,189.01
0,A,84,103.25


In [68]:
# ok I found this on page 16:https://www.rrc.texas.gov/media/kmld3uzj/digital-map-information-user-guide.pdf

In [69]:
# I define the official RRC pipeline status descriptions.
# I preserve the original codes for traceability.

status_labels = {
    "I": "In Service",
    "B": "Abandoned",
    "R": "Retired"
}

# I define the RRC pipeline system-type descriptions.

system_labels = {
    "G": "Gas Gathering",
    "T": "Gas Transmission",
    "Q": "Highly Volatile Liquid",
    "O": "Crude Oil Transmission",
    "L": "Crude Oil Gathering",
    "P": "Refined Liquid Product",
    "K": "Carbon Dioxide",
    "Z": "Gas Offshore",
    "C": "Anhydrous Ammonia",
    "A": "Crude Oil Offshore"
}

# I add readable descriptions without replacing the original codes.

texas_pipelines_clean["status_description"] = (
    texas_pipelines_clean["STATUS_CD"].map(status_labels)
)

texas_pipelines_clean["system_description"] = (
    texas_pipelines_clean["SYSTYPE"].map(system_labels)
)

# I check for codes that are missing from my dictionaries.
# I want to avoid silently losing classifications.

print(
    "Unmapped status records:",
    texas_pipelines_clean["status_description"].isna().sum()
)

print(
    "Unmapped system-type records:",
    texas_pipelines_clean["system_description"].isna().sum()
)

# I inspect the results.

display(
    texas_pipelines_clean[
        ["STATUS_CD", "status_description",
         "SYSTYPE", "system_description"]
    ].head()
)

Unmapped status records: 0
Unmapped system-type records: 0


,STATUS_CD,status_description,SYSTYPE,system_description
0,I,In Service,G,Gas Gathering
1,B,Abandoned,G,Gas Gathering
2,I,In Service,T,Gas Transmission
3,I,In Service,T,Gas Transmission
4,I,In Service,T,Gas Transmission


In [70]:
# saving file

In [71]:
# I define the folder where I save my cleaned datasets.

cleaned_data_path = file_path + "03_Cleaned_Data/"

# I create a separate dataset containing only in-service pipelines.
# I keep all statuses in the original cleaned dataset
# because abandoned and retired pipelines may still affect land.

texas_pipelines_in_service = texas_pipelines_clean[
    texas_pipelines_clean["STATUS_CD"] == "I"
].copy()

# I save all cleaned pipeline records as a GeoPackage.
# I preserve their geometries for spatial analysis in Notebook 02.

texas_pipelines_clean.to_file(
    cleaned_data_path + "Texas_Pipelines_All_Statuses.gpkg",
    layer="texas_pipelines",
    driver="GPKG"
)

# I save the in-service pipelines as a separate GeoPackage.

texas_pipelines_in_service.to_file(
    cleaned_data_path + "Texas_Pipelines_In_Service.gpkg",
    layer="texas_pipelines",
    driver="GPKG"
)

# I create CSV versions for sharing and statistical analysis.
# I remove the geometry column because CSV does not support
# spatial geometries directly.
# I retain geometry_miles and all original RRC attributes.

texas_pipelines_all_csv = texas_pipelines_clean.drop(
    columns="geometry"
)

texas_pipelines_in_service_csv = texas_pipelines_in_service.drop(
    columns="geometry"
)

# I save both CSV files.

texas_pipelines_all_csv.to_csv(
    cleaned_data_path + "Texas_Pipelines_All_Statuses.csv",
    index=False
)

texas_pipelines_in_service_csv.to_csv(
    cleaned_data_path + "Texas_Pipelines_In_Service.csv",
    index=False
)

# I confirm that all four files have been saved.

print("All-status pipeline GeoPackage and CSV saved.")
print("In-service pipeline GeoPackage and CSV saved.")

print("Total cleaned pipeline records:", len(texas_pipelines_clean))
print("In-service pipeline records:", len(texas_pipelines_in_service))

print(
    "Total cleaned pipeline miles:",
    round(texas_pipelines_clean["geometry_miles"].sum(), 2)
)

print(
    "In-service pipeline miles:",
    round(texas_pipelines_in_service["geometry_miles"].sum(), 2)
)

All-status pipeline GeoPackage and CSV saved.
In-service pipeline GeoPackage and CSV saved.
Total cleaned pipeline records: 496305
In-service pipeline records: 435979
Total cleaned pipeline miles: 363844.11
In-service pipeline miles: 315158.02


In [72]:
# Refinery data downlaoded for 2026 here:https://www.eia.gov/petroleum/refinerycapacity/index.php

In [73]:
# Underground disposal wells: #PRC-UIC well location downloaded from: https://data.texas.gov/dataset/RRC-UIC-well-location/givw-z9t4/about_data

In [74]:
#Fossil fuel power plant for 2025 downloaded:https://www.eia.gov/petroleum/refinerycapacity/index.php

In [75]:
# Tony said to add these data :

### 1.7 Loading fossil fuel power plants

In [76]:
#source:https://www.eia.gov/electricity/data/eia860/
# I reuse the EIA-860 plant and generator datasets
# that I already loaded for my battery analysis.

fossil_plants_raw = battery_plants.copy()
fossil_generators_raw = battery_generators.copy()

# I check the number of records before filtering.
print("EIA plant records:", len(fossil_plants_raw))
print("EIA generator records:", len(fossil_generators_raw))

# I inspect the available columns to identify fuel types,
# operating status, capacity, and geographic information.
print("\nPlant columns:")
print(fossil_plants_raw.columns.tolist())

print("\nGenerator columns:")
print(fossil_generators_raw.columns.tolist())

# I inspect the primary fuel categories.
fuel_columns = [
    col for col in fossil_generators_raw.columns
    if "energy source" in col.lower()
    or "technology" in col.lower()
    or "status" in col.lower()
]

for col in fuel_columns:
    print(f"\n{col}:")
    print(fossil_generators_raw[col].value_counts(dropna=False).head(30))


EIA plant records: 17349
EIA generator records: 27919

Plant columns:
['Utility ID', 'Utility Name', 'Plant Code', 'Plant Name', 'Street Address', 'City', 'State', 'Zip', 'County', 'Latitude', 'Longitude', 'NERC Region', 'Balancing Authority Code', 'Balancing Authority Name', 'Name of Water Source', 'Primary Purpose (NAICS Code)', 'Regulatory Status', 'Sector', 'Sector Name', 'FERC Cogeneration Status', 'FERC Cogeneration Docket Number', 'FERC Small Power Producer Status', 'FERC Small Power Producer Docket Number', 'FERC Exempt Wholesale Generator Status', 'FERC Exempt Wholesale Generator Docket Number', 'Ash Impoundment?', 'Ash Impoundment Lined?', 'Ash Impoundment Status', 'Transmission or Distribution System Owner', 'Transmission or Distribution System Owner ID', 'Transmission or Distribution System Owner State', 'Grid Voltage (kV)', 'Grid Voltage 2 (kV)', 'Grid Voltage 3 (kV)', 'Energy Storage', 'Natural Gas LDC Name', 'Natural Gas Pipeline Name 1', 'Natural Gas Pipeline Name 2', '

## fossil fuel power plants

In [77]:
# I define the fossil fuel codes using EIA energy source classifications.
# I include natural gas, coal, petroleum, and fossil-derived gases.

fossil_fuel_codes = {
    # Natural gas
    "NG",

    # Coal
    "BIT", "SUB", "LIG", "WC", "RC", "ANT",

    # Petroleum and petroleum-derived fuels
    "DFO", "RFO", "KER", "JF", "PC", "WO",

    # Other fossil-derived gases
    "OG", "BFG", "PG", "SGC"
}

# I select operating generators located in Texas.
# I use the primary energy source to avoid counting generators
# that use fossil fuels only as secondary or backup fuels.

tx_fossil_generators = fossil_generators_raw.loc[
    (fossil_generators_raw["State"] == "TX") &
    (fossil_generators_raw["Status"] == "OP") &
    (fossil_generators_raw["Energy Source 1"].isin(fossil_fuel_codes))
].copy()

# I convert nameplate capacity to numeric values.
tx_fossil_generators["Nameplate Capacity (MW)"] = pd.to_numeric(
    tx_fossil_generators["Nameplate Capacity (MW)"],
    errors="coerce"
)

print("Operating fossil fuel generators in Texas:",
      len(tx_fossil_generators))

print("\nGenerators by technology:")
print(tx_fossil_generators["Technology"].value_counts())

print("\nGenerators by primary fuel:")
print(tx_fossil_generators["Energy Source 1"].value_counts())

print("\nTotal nameplate capacity (MW):")
print(
    tx_fossil_generators["Nameplate Capacity (MW)"].sum()
)

print("\nUnique power plants:")
print(tx_fossil_generators["Plant Code"].nunique())

Operating fossil fuel generators in Texas: 1565

Generators by technology:
Technology
Natural Gas Internal Combustion Engine    667
Petroleum Liquids                         289
Natural Gas Fired Combined Cycle          278
Natural Gas Fired Combustion Turbine      227
Natural Gas Steam Turbine                  70
Conventional Steam Coal                    24
Other Gases                                 8
Petroleum Coke                              2
Name: count, dtype: int64

Generators by primary fuel:
Energy Source 1
NG     1242
DFO     289
SUB      19
OG        7
LIG       5
PC        2
BFG       1
Name: count, dtype: int64

Total nameplate capacity (MW):
97326.29999999999

Unique power plants:
342


In [78]:
# I check which electricity sectors contain fossil fuel generators.
print("Generators by sector:")
print(tx_fossil_generators["Sector Name"].value_counts())

# I examine generator capacity by technology to identify
# smaller facilities and utility-scale power generation.
print("\nCapacity by technology:")

display(
    tx_fossil_generators
    .groupby("Technology")
    .agg(
        generators=("Generator ID", "count"),
        total_capacity_mw=("Nameplate Capacity (MW)", "sum"),
        median_capacity_mw=("Nameplate Capacity (MW)", "median")
    )
    .sort_values("total_capacity_mw", ascending=False)
    .round(2)
)

# I check whether petroleum generators are primarily small units.
print("\nPetroleum generator capacity:")
print(
    tx_fossil_generators.loc[
        tx_fossil_generators["Energy Source 1"] == "DFO",
        "Nameplate Capacity (MW)"
    ].describe()
)

Generators by sector:
Sector Name
IPP Non-CHP           838
Electric Utility      270
Commercial Non-CHP    247
Industrial CHP        118
IPP CHP                55
Industrial Non-CHP     25
Commercial CHP         12
Name: count, dtype: int64

Capacity by technology:


,generators,total_capacity_mw,median_capacity_mw
Technology,,,
Natural Gas Fired Combined Cycle,278,47038.4,176.00
Natural Gas Steam Turbine,70,16931.8,189.10
Conventional Steam Coal,24,15573.5,618.70
Natural Gas Fired Combustion Turbine,227,15407.4,60.50
Natural Gas Internal Combustion Engine,667,1322.2,0.40
Petroleum Liquids,289,788.5,0.60
Other Gases,8,207.3,20.25
Petroleum Coke,2,57.2,28.60



Petroleum generator capacity:
count    289.000000
mean       2.728374
std       12.967878
min        0.600000
25%        0.600000
50%        0.600000
75%        0.600000
max       89.400000
Name: Nameplate Capacity (MW), dtype: float64


In [79]:
# METHODOLOGICAL DECISIONS
# I include both small and large fossil fuel generation facilities
# because my objective is to map the distribution of fossil fuel
# infrastructure across Texas, not only utility-scale power plants.
#
# I recognize that small diesel generators and large natural gas
# or coal power plants differ substantially in generating capacity.
# Therefore, I preserve their capacity and sector information
# rather than treating all facilities as equivalent in analysis.
#
# I use the EIA Plant Code as the facility identifier because
# multiple generators can belong to the same power plant.
# I aggregate generators to one record per Plant Code to avoid
# displaying multiple points for the same facility.
#
# I calculate fossil capacity using only operating generators
# whose primary energy source is fossil fuel. This excludes
# fossil fuels reported only as secondary or backup sources.
#
# I preserve all primary fossil fuel categories and electricity
# sectors associated with each plant instead of selecting only
# the first generator's classification.

import pandas as pd
import geopandas as gpd
import os

# I define broad fuel categories for the selected generators.
fuel_groups = {
    "NG": "Natural Gas",
    "BIT": "Coal",
    "SUB": "Coal",
    "LIG": "Coal",
    "WC": "Coal",
    "RC": "Coal",
    "ANT": "Coal",
    "DFO": "Petroleum",
    "RFO": "Petroleum",
    "KER": "Petroleum",
    "JF": "Petroleum",
    "PC": "Petroleum",
    "WO": "Petroleum",
    "OG": "Other Fossil Gas",
    "BFG": "Other Fossil Gas",
    "PG": "Other Fossil Gas",
    "SGC": "Other Fossil Gas"
}

# I create a working copy to preserve the original generator data.
tx_fossil_working = tx_fossil_generators.copy()

tx_fossil_working["fuel_category"] = (
    tx_fossil_working["Energy Source 1"].map(fuel_groups)
)

# I check that every selected fuel code has a classification.
unclassified = tx_fossil_working[
    tx_fossil_working["fuel_category"].isna()
]

print("Unclassified generators:", len(unclassified))

if len(unclassified) > 0:
    raise ValueError(
        "Some fossil fuel codes are not classified."
    )

# I check the uniqueness of generator identifiers within plants.
duplicate_generators = tx_fossil_working.duplicated(
    subset=["Plant Code", "Generator ID"]
).sum()

print("Duplicate generator identifiers:", duplicate_generators)

if duplicate_generators > 0:
    raise ValueError(
        "Duplicate plant-generator combinations require review."
    )

# I aggregate operating fossil fuel generators by plant.
tx_fossil_plants = (
    tx_fossil_working
    .groupby("Plant Code", as_index=False)
    .agg(
        plant_name=("Plant Name", "first"),
        utility_name=("Utility Name", "first"),
        county=("County", "first"),
        generator_count=("Generator ID", "nunique"),
        fossil_capacity_mw=("Nameplate Capacity (MW)", "sum"),
        fuel_types=(
            "fuel_category",
            lambda x: ", ".join(sorted(x.dropna().unique()))
        ),
        sectors=(
            "Sector Name",
            lambda x: ", ".join(sorted(x.dropna().unique()))
        )
    )
)

print("\nTotal fossil fuel power plants:", len(tx_fossil_plants))
print("Total generators:", tx_fossil_plants["generator_count"].sum())
print("Total fossil capacity (MW):",
      round(tx_fossil_plants["fossil_capacity_mw"].sum(), 2))

print("\nPlants by fuel combination:")
print(tx_fossil_plants["fuel_types"].value_counts())

display(tx_fossil_plants.head())

Unclassified generators: 0
Duplicate generator identifiers: 0

Total fossil fuel power plants: 342
Total generators: 1565
Total fossil capacity (MW): 97326.3

Plants by fuel combination:
fuel_types
Natural Gas                      297
Petroleum                         25
Coal                              10
Natural Gas, Petroleum             4
Natural Gas, Other Fossil Gas      3
Other Fossil Gas                   2
Coal, Natural Gas                  1
Name: count, dtype: int64


,Plant Code,plant_name,utility_name,county,generator_count,fossil_capacity_mw,fuel_types,sectors
0,9.0,Copper,El Paso Electric Co,El Paso,1,86.9,Natural Gas,Electric Utility
1,298.0,Limestone,NRG Texas Power LLC,Limestone,2,1849.8,Coal,IPP Non-CHP
2,3439.0,Laredo,City of San Antonio - (TX),Webb,2,263.6,Natural Gas,Electric Utility
3,3441.0,Nueces Bay,City of San Antonio - (TX),Nueces,3,730.2,Natural Gas,Electric Utility
4,3443.0,Victoria,"Victoria WLE, LP",Victoria,2,376.9,Natural Gas,IPP Non-CHP


In [80]:
# I preserve all 342 plant records during the merge, even if
# some coordinates are missing. I do not want missing geographic
# information to silently remove facilities from the inventory.


# I standardize the plant identifiers in both datasets.
tx_fossil_plants["Plant Code"] = pd.to_numeric(
    tx_fossil_plants["Plant Code"], errors="raise"
).astype("int64")

tx_plant_locations = fossil_plants_raw.loc[
    fossil_plants_raw["State"] == "TX",
    [
        "Plant Code",
        "Latitude",
        "Longitude",
        "City"
    ]
].copy()

tx_plant_locations["Plant Code"] = pd.to_numeric(
    tx_plant_locations["Plant Code"], errors="raise"
).astype("int64")

# I check for duplicate plant identifiers before merging.
duplicate_locations = tx_plant_locations[
    tx_plant_locations.duplicated("Plant Code", keep=False)
]

print("Duplicate plant location records:",
      len(duplicate_locations))

if len(duplicate_locations) > 0:
    display(duplicate_locations.head(20))
    raise ValueError(
        "Duplicate plant codes require investigation before merging."
    )

# I merge the coordinates without removing any fossil fuel plants.
tx_fossil_plants_geo = tx_fossil_plants.merge(
    tx_plant_locations,
    on="Plant Code",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("\nMerge results:")
print(tx_fossil_plants_geo["_merge"].value_counts())

# I convert coordinates to numeric values.
tx_fossil_plants_geo["Latitude"] = pd.to_numeric(
    tx_fossil_plants_geo["Latitude"],
    errors="coerce"
)

tx_fossil_plants_geo["Longitude"] = pd.to_numeric(
    tx_fossil_plants_geo["Longitude"],
    errors="coerce"
)

# I identify missing coordinates.
missing_coordinates = (
    tx_fossil_plants_geo[["Latitude", "Longitude"]]
    .isna()
    .any(axis=1)
)

# I identify coordinates outside a broad Texas geographic range.
valid_coordinates = (
    tx_fossil_plants_geo["Latitude"].between(25, 37) &
    tx_fossil_plants_geo["Longitude"].between(-107, -93)
)

print("\nTotal fossil fuel plants:",
      len(tx_fossil_plants_geo))

print("Plants missing coordinates:",
      missing_coordinates.sum())

print("Plants with coordinates in Texas bounding range:",
      valid_coordinates.sum())

print("Plants requiring geographic review:",
      (~valid_coordinates).sum())

# I inspect problematic records instead of deleting them.
if (~valid_coordinates).sum() > 0:
    display(
        tx_fossil_plants_geo.loc[
            ~valid_coordinates,
            [
                "Plant Code",
                "plant_name",
                "county",
                "City",
                "Latitude",
                "Longitude"
            ]
        ]
    )

# I verify that the geographic merge preserved the original inventory.
assert len(tx_fossil_plants_geo) == len(tx_fossil_plants), (
    "The geographic merge changed the number of fossil fuel plants."
)

display(tx_fossil_plants_geo.head())

Duplicate plant location records: 0

Merge results:
_merge
both          342
left_only       0
right_only      0
Name: count, dtype: int64

Total fossil fuel plants: 342
Plants missing coordinates: 0
Plants with coordinates in Texas bounding range: 342
Plants requiring geographic review: 0


,Plant Code,plant_name,utility_name,county,generator_count,fossil_capacity_mw,fuel_types,sectors,Latitude,Longitude,City,_merge
0,9,Copper,El Paso Electric Co,El Paso,1,86.9,Natural Gas,Electric Utility,31.756900,-106.375000,El Paso,both
1,298,Limestone,NRG Texas Power LLC,Limestone,2,1849.8,Coal,IPP Non-CHP,31.421900,-96.252500,Jewett,both
2,3439,Laredo,City of San Antonio - (TX),Webb,2,263.6,Natural Gas,Electric Utility,27.566700,-99.508900,Laredo,both
3,3441,Nueces Bay,City of San Antonio - (TX),Nueces,3,730.2,Natural Gas,Electric Utility,27.819412,-97.419203,Corpus Christi,both
4,3443,Victoria,"Victoria WLE, LP",Victoria,2,376.9,Natural Gas,IPP Non-CHP,28.788300,-97.010000,Victoria,both


In [81]:
# I save every fossil power plant in a nonspatial inventory, including records
# whose coordinates are missing or outside the broad Texas coordinate range.
fossil_output_folder = os.path.join(file_path, "03_Cleaned_Data", "Fossil_Power_Plants")
os.makedirs(fossil_output_folder, exist_ok=True)
tx_fossil_export = tx_fossil_plants_geo.drop(columns=["_merge"]).copy()
tx_fossil_export["coordinate_review_needed"] = ~valid_coordinates
tx_fossil_export.to_csv(
    os.path.join(fossil_output_folder, "TX_Fossil_Power_Plants_Inventory.csv"), index=False
)

# I create map geometry only for facilities with plausible Texas coordinates.
# I do not interpret a bounding-box check as proof of a correct facility location.
tx_fossil_map = tx_fossil_export.loc[valid_coordinates].copy()
tx_fossil_gdf = gpd.GeoDataFrame(
    tx_fossil_map,
    geometry=gpd.points_from_xy(tx_fossil_map["Longitude"], tx_fossil_map["Latitude"]),
    crs="EPSG:4326"
)
fossil_output_path = os.path.join(fossil_output_folder, "TX_Fossil_Power_Plants.gpkg")
tx_fossil_gdf.to_file(fossil_output_path, layer="fossil_power_plants", driver="GPKG")
fossil_check = gpd.read_file(fossil_output_path)
assert len(fossil_check) == valid_coordinates.sum()
assert len(tx_fossil_export) == len(tx_fossil_plants)
assert tx_fossil_export["Plant Code"].is_unique
print("Complete fossil plant inventory:", len(tx_fossil_export))
print("Map-ready fossil plants:", len(fossil_check))
print("Facilities requiring coordinate review:", (~valid_coordinates).sum())


Complete fossil plant inventory: 342
Map-ready fossil plants: 342
Facilities requiring coordinate review: 0


## Loading fossil fuels Refinery

In [82]:
# I use the existing EIA_Refinery folder to maintain a consistent
# project structure and avoid creating duplicate directories.
#
# I preserve the original EIA refinery workbook without modifying
# its contents. I first inspect the available worksheets because
# the refinery inventory may contain multiple tables and headers.
# 
# I will include both small and large refineries in the analysis
# and retain their processing capacities to distinguish their scale.

import os
import pandas as pd

# I use the existing refinery folder.
refinery_folder = os.path.join(
    raw_data_path,
    "EIA_Refinery"
)

print("Refinery folder:", refinery_folder)
print("Folder exists:", os.path.exists(refinery_folder))

# I check which files are available.
refinery_files = [
    f for f in os.listdir(refinery_folder)
    if f.lower().endswith((".xlsx", ".xls"))
    and not f.startswith("~$")
]

print("\nAvailable refinery files:")
print(refinery_files)

if len(refinery_files) == 0:
    raise FileNotFoundError(
        "No Excel refinery workbook found in EIA_Refinery."
    )

# I inspect each workbook rather than assuming a specific filename.
for filename in refinery_files:

    refinery_path = os.path.join(refinery_folder, filename)
    refinery_excel = pd.ExcelFile(refinery_path)

    print(f"\nWorkbook: {filename}")
    print("Available worksheets:", refinery_excel.sheet_names)

    for sheet in refinery_excel.sheet_names:

        print(f"\nWorksheet: {sheet}")

        preview = pd.read_excel(
            refinery_path,
            sheet_name=sheet,
            header=None,
            nrows=12
        )

        display(preview)

Refinery folder: G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint/01_Raw_Data/EIA_Refinery
Folder exists: True

Available refinery files:
['refcap26.xlsx']

Workbook: refcap26.xlsx
Available worksheets: ['refcap26']

Worksheet: refcap26


,0,1,2,3,4,5,6,7,8,9,10
0,CORPORATION,SURVEY,PERIOD,COMPANY_NAME,RDIST_LABEL,STATE_NAME,SITE,PADD,PRODUCT,SUPPLY,QUANTITY
1,ACCESS INDUSTRIES,820,26,EQUISTAR CHEMICALS LP,Texas Gulf Coast,Texas,CHANNELVIEW,3,ALKYLATES,"Production Capacity, Current Year (barrels per...",22370
2,ACCESS INDUSTRIES,820,26,EQUISTAR CHEMICALS LP,Texas Gulf Coast,Texas,CHANNELVIEW,3,ALKYLATES,"Production Capacity, Next Year (barrels per st...",22370
3,AMERICAN REFINING GROUP INC,820,26,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,ASPHALT & ROAD OIL,"Production Capacity, Current Year (barrels per...",560
4,AMERICAN REFINING GROUP INC,820,26,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,ASPHALT & ROAD OIL,"Production Capacity, Next Year (barrels per st...",560
5,AMERICAN REFINING GROUP INC,820,26,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,CAT REFORMING: HIGH PRESSURE,"Downstream Charge Capacity, Current Year (barr...",2050
6,AMERICAN REFINING GROUP INC,820,26,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,CAT REFORMING: HIGH PRESSURE,"Downstream Charge Capacity, Current Year (barr...",2200
7,AMERICAN REFINING GROUP INC,820,26,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,CAT REFORMING: HIGH PRESSURE,"Downstream Charge Capacity, Next Year (barrels...",2200
8,AMERICAN REFINING GROUP INC,820,26,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,"DESULFURIZATION, DIESEL FUEL","Downstream Charge Capacity, Current Year (barr...",4800
9,AMERICAN REFINING GROUP INC,820,26,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,"DESULFURIZATION, DIESEL FUEL","Downstream Charge Capacity, Next Year (barrels...",4800


In [83]:
# I do not sum all capacity values because they represent
# different processes, products, and reporting periods.
# Adding them together would overestimate refinery capacity.
# I will identify the crude oil distillation capacity measure
# before comparing refinery sizes.

# I load the refinery worksheet using its first row as the header.
# I select the workbook explicitly rather than relying on the final iteration of a loop.
if len(refinery_files) != 1:
    raise ValueError("I need to select and verify the correct refinery workbook explicitly.")
refinery_path = os.path.join(refinery_folder, refinery_files[0])
refinery_raw = pd.read_excel(
    refinery_path,
    sheet_name="refcap26"
)

print("Total refinery capacity records:",
      len(refinery_raw))

print("\nAvailable columns:")
print(refinery_raw.columns.tolist())

# I select Texas refinery records.
tx_refinery_raw = refinery_raw.loc[
    refinery_raw["STATE_NAME"]
    .astype(str)
    .str.strip()
    .str.upper() == "TEXAS"
].copy()

print("\nTexas capacity records:",
      len(tx_refinery_raw))

# I identify unique refinery sites using company and site names.
refinery_sites = (
    tx_refinery_raw[
        ["COMPANY_NAME", "SITE", "STATE_NAME"]
    ]
    .drop_duplicates()
    .sort_values(["SITE", "COMPANY_NAME"])
    .reset_index(drop=True)
)

print("\nPreliminary number of Texas refinery sites:",
      len(refinery_sites))

display(refinery_sites)

# I inspect the different products and capacity measures
# before selecting a consistent refinery capacity indicator.
print("\nAvailable product categories:")
display(
    tx_refinery_raw["PRODUCT"]
    .value_counts(dropna=False)
)

print("\nAvailable capacity measures:")
display(
    tx_refinery_raw["SUPPLY"]
    .value_counts(dropna=False)
)

Total refinery capacity records: 3336

Available columns:
['CORPORATION', 'SURVEY', 'PERIOD', 'COMPANY_NAME', 'RDIST_LABEL', 'STATE_NAME', 'SITE', 'PADD', 'PRODUCT', 'SUPPLY', 'QUANTITY']

Texas capacity records: 773

Preliminary number of Texas refinery sites: 34


,COMPANY_NAME,SITE,STATE_NAME
0,EXXONMOBIL REFINING & SUPPLY CO,BAYTOWN,Texas
1,EXXONMOBIL REFINING & SUPPLY CO,BEAUMONT,Texas
2,ALON USA ENERGY INC,BIG SPRING,Texas
3,WRB REFINING LP,BORGER,Texas
4,EQUISTAR CHEMICALS LP,CHANNELVIEW,Texas
5,HARTREE REFINING LLC,CHANNELVIEW,Texas
6,BUCKEYE TEXAS PROCESSING LLC,CORPUS CHRISTI,Texas
7,CITGO REFINING & CHEMICAL INC,CORPUS CHRISTI,Texas
8,MAGELLAN PROCESSING LP,CORPUS CHRISTI,Texas
9,VALERO REFINING CO TEXAS LP,CORPUS CHRISTI,Texas



Available product categories:


PRODUCT
CAT CRACKING: FRESH FEED                  60
OPERATING CAPACITY                        58
TOTAL OPERABLE CAPACITY                   58
CAT REFORMING: LOW PRESSURE               54
DESULFURIZATION, NAPHTHA/REFORMER FEED    46
ALKYLATES                                 42
VACUUM DISTILLATION                       40
THERM CRACKING, DELAYED COKING            39
SULFUR (SHORT TONS/DAY)                   38
DESULFURIZATION, DIESEL FUEL              38
DESULFURIZATION, GASOLINE                 32
DESULFURIZATION, KEROSENE AND JET         30
TOTAL OPER CAP (PROJECTED, NEXT YEAR)     29
DESULFURIZATION, HEAVY GAS OIL            26
PETCOKE,MARKET                            26
CAT HYDROCRACKING, GAS OIL                21
CAT REFORMING: HIGH PRESSURE              18
DESULFURIZATION, OTHER                    18
AROMATICS                                 16
FUELS SOLVENT DEASPHALTING                14
ISOMERIZATION (ISOPENTANE/ISOHEXANE)      14
CAT HYDROCRACKING, DISTILLATE             12
IS


Available capacity measures:


SUPPLY
Downstream Charge Capacity, Next Year (barrels per stream day)                          195
Downstream Charge Capacity, Current Year (barrels per stream day)                       195
Atmospheric Crude Distillation Capacity (barrels per stream day)                         87
Production Capacity, Current Year (barrels per steam day except sulfur and hydrogen)     84
Production Capacity, Next Year (barrels per stream day except sulfur and hydrogen)       84
Downstream Charge Capacity, Current Year (barrels per calendar day)                      70
Atmospheric Crude Distillation Capacity (barrels per calendar day)                       58
Name: count, dtype: int64

In [84]:
# I use atmospheric crude distillation capacity in barrels per
# calendar day as the primary indicator of refinery processing scale.
# I choose calendar-day capacity because it accounts for expected
# downtime and provides a consistent measure across facilities.
# I do not sum different refining processes because this would
# double count processing capacity within the same refinery.
# I inspect records before aggregation because multiple capacity
# observations may belong to the same refinery.

# I select atmospheric crude distillation capacity records.
tx_refinery_capacity = tx_refinery_raw.loc[
    tx_refinery_raw["SUPPLY"].astype(str).str.contains(
        "Atmospheric Crude Distillation Capacity",
        case=False,
        na=False
    )
].copy()

print("Total atmospheric distillation capacity records:",
      len(tx_refinery_capacity))

# I examine which product categories are associated
# with atmospheric crude distillation capacity.
print("\nCapacity records by product and measurement:")
display(
    tx_refinery_capacity
    .groupby(["PRODUCT", "SUPPLY"])
    .size()
    .reset_index(name="records")
    .sort_values(["PRODUCT", "SUPPLY"])
)

# I examine the number of distinct company-site combinations.
print("\nUnique refinery company-site combinations:")
print(
    tx_refinery_capacity[
        ["COMPANY_NAME", "SITE", "STATE_NAME"]
    ].drop_duplicates().shape[0]
)

# I inspect the capacity records to identify possible duplicates.
display(
    tx_refinery_capacity[
        [
            "COMPANY_NAME",
            "SITE",
            "PRODUCT",
            "SUPPLY",
            "QUANTITY"
        ]
    ]
    .sort_values(["SITE", "COMPANY_NAME", "PRODUCT", "SUPPLY"])
    .head(40)
)

Total atmospheric distillation capacity records: 145

Capacity records by product and measurement:


,PRODUCT,SUPPLY,records
0,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,29
1,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,29
2,"TOTAL OPER CAP (PROJECTED, NEXT YEAR)",Atmospheric Crude Distillation Capacity (barre...,29
3,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,29
4,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,29



Unique refinery company-site combinations:
29


,COMPANY_NAME,SITE,PRODUCT,SUPPLY,QUANTITY
909,EXXONMOBIL REFINING & SUPPLY CO,BAYTOWN,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,564500
910,EXXONMOBIL REFINING & SUPPLY CO,BAYTOWN,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,588000
921,EXXONMOBIL REFINING & SUPPLY CO,BAYTOWN,"TOTAL OPER CAP (PROJECTED, NEXT YEAR)",Atmospheric Crude Distillation Capacity (barre...,588000
922,EXXONMOBIL REFINING & SUPPLY CO,BAYTOWN,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,564500
923,EXXONMOBIL REFINING & SUPPLY CO,BAYTOWN,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,588000
951,EXXONMOBIL REFINING & SUPPLY CO,BEAUMONT,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,612000
952,EXXONMOBIL REFINING & SUPPLY CO,BEAUMONT,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,637400
960,EXXONMOBIL REFINING & SUPPLY CO,BEAUMONT,"TOTAL OPER CAP (PROJECTED, NEXT YEAR)",Atmospheric Crude Distillation Capacity (barre...,637400
961,EXXONMOBIL REFINING & SUPPLY CO,BEAUMONT,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,612000
962,EXXONMOBIL REFINING & SUPPLY CO,BEAUMONT,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,637400


In [85]:
# I select the two current-year operating capacity measures.
operating_capacity = tx_refinery_capacity.loc[
    tx_refinery_capacity["PRODUCT"].eq("OPERATING CAPACITY")
].copy()

# I classify capacity units explicitly.
operating_capacity["capacity_type"] = pd.NA

operating_capacity.loc[
    operating_capacity["SUPPLY"].str.contains(
        "barrels per calendar day", case=False, na=False
    ),
    "capacity_type"
] = "calendar_day"

operating_capacity.loc[
    operating_capacity["SUPPLY"].str.contains(
        "barrels per stream day", case=False, na=False
    ),
    "capacity_type"
] = "stream_day"

# I check that each facility has one record for each capacity type.
facility_columns = [
    "COMPANY_NAME",
    "SITE",
    "STATE_NAME"
]

capacity_counts = (
    operating_capacity
    .groupby(facility_columns + ["capacity_type"])
    .size()
)

print("Capacity records per facility and type:")
print(capacity_counts.value_counts())

if not capacity_counts.eq(1).all():
    raise ValueError(
        "Duplicate capacity records require investigation."
    )

# I convert the reported capacity to numeric values.
operating_capacity["QUANTITY"] = pd.to_numeric(
    operating_capacity["QUANTITY"],
    errors="coerce"
)

# I reshape the data to one row per refinery.
tx_refineries = (
    operating_capacity
    .pivot(
        index=facility_columns,
        columns="capacity_type",
        values="QUANTITY"
    )
    .reset_index()
    .rename(
        columns={
            "calendar_day": "capacity_bpcd",
            "stream_day": "capacity_bpsd",
            "COMPANY_NAME": "company_name",
            "SITE": "site",
            "STATE_NAME": "state"
        }
    )
)

# I verify that both capacity measures are available.
print("\nTotal Texas refineries:", len(tx_refineries))

print("Missing calendar-day capacities:",
      tx_refineries["capacity_bpcd"].isna().sum())

print("Missing stream-day capacities:",
      tx_refineries["capacity_bpsd"].isna().sum())

print("\nTotal operating refinery capacity (b/cd):")
print(tx_refineries["capacity_bpcd"].sum())

print("\nRefineries by capacity:")
display(
    tx_refineries.sort_values(
        "capacity_bpcd", ascending=False
    )
)

assert tx_refineries["capacity_bpcd"].notna().all()
assert tx_refineries["capacity_bpsd"].notna().all()

Capacity records per facility and type:
1    58
Name: count, dtype: int64

Total Texas refineries: 29
Missing calendar-day capacities: 0
Missing stream-day capacities: 0

Total operating refinery capacity (b/cd):
6134650

Refineries by capacity:


capacity_type,company_name,site,state,capacity_bpcd,capacity_bpsd
16,MOTIVA ENTERPRISES LLC,PORT ARTHUR,Texas,656400,678000
15,MARATHON PETROLEUM CO LP,GALVESTON BAY,Texas,631000,665000
8,EXXONMOBIL REFINING & SUPPLY CO,BEAUMONT,Texas,612000,637400
7,EXXONMOBIL REFINING & SUPPLY CO,BAYTOWN,Texas,564500,588000
20,PREMCOR REFINING GROUP INC,PORT ARTHUR,Texas,385000,424000
3,DEER PARK REFINING LP,DEER PARK,Texas,312500,340000
24,VALERO REFINING CO TEXAS LP,CORPUS CHRISTI,Texas,290000,301000
10,FLINT HILLS RESOURCES LP,CORPUS CHRISTI WEST,Texas,290000,300000
19,PHILLIPS 66 COMPANY,SWEENY,Texas,277000,291700
23,TOTALENERGIES PETROCHEM & REFG USA,PORT ARTHUR,Texas,238000,245000


In [86]:
#I assign a stable identifier to each refinery in my EIA inventory.
# I retain company and site names separately because several
# refineries can operate within the same city.
# I export a reference inventory so I can document which
# facilities are matched to verified geographic locations.
# I sort the facilities to make the identifiers reproducible.
tx_refineries = (
    tx_refineries
    .sort_values(["site", "company_name"])
    .reset_index(drop=True)
)

# I assign one identifier per preliminary refinery facility.
tx_refineries.insert(
    0,
    "refinery_id",
    [
        f"TX_REF_{i:03d}"
        for i in range(1, len(tx_refineries) + 1)
    ]
)

# I create the cleaned refinery output directory.
refinery_output_folder = os.path.join(
    file_path,
    "03_Cleaned_Data",
    "Refineries"
)

os.makedirs(refinery_output_folder, exist_ok=True)

# I save the inventory before geographic matching.
refinery_inventory_path = os.path.join(
    refinery_output_folder,
    "TX_Refinery_Inventory.csv"
)

tx_refineries.to_csv(
    refinery_inventory_path,
    index=False
)

# I verify that the exported inventory contains unique IDs.
assert tx_refineries["refinery_id"].is_unique
print("Saved refinery inventory:", refinery_inventory_path)
print("Total refineries:", len(tx_refineries))
print("Unique refinery IDs:", tx_refineries["refinery_id"].nunique())

display(tx_refineries.head())

Saved refinery inventory: G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint/03_Cleaned_Data\Refineries\TX_Refinery_Inventory.csv
Total refineries: 29
Unique refinery IDs: 29


capacity_type,refinery_id,company_name,site,state,capacity_bpcd,capacity_bpsd
0,TX_REF_001,EXXONMOBIL REFINING & SUPPLY CO,BAYTOWN,Texas,564500,588000
1,TX_REF_002,EXXONMOBIL REFINING & SUPPLY CO,BEAUMONT,Texas,612000,637400
2,TX_REF_003,ALON USA ENERGY INC,BIG SPRING,Texas,73000,74000
3,TX_REF_004,WRB REFINING LP,BORGER,Texas,149000,165542
4,TX_REF_005,HARTREE REFINING LLC,CHANNELVIEW,Texas,45000,50000
